# Array max: block reduction plus one atomic per block

The atomic sum in this plan issues one atomic per **thread**. When thousands of threads hit the same address, the hardware serialises them, so a better pattern is: reduce inside each block first (cheap, in shared memory), then let **one thread per block** publish the block's result with a single atomic.

Write `array_max(x)` that returns the largest element of a 1-D `float32` array.

## Requirements
- Fixed grid: **8 blocks of 64 threads**, grid-stride loop, so the grid does not depend on `n`.
- Each thread folds its elements into a local maximum, writes it to a 64-entry `float32` shared array, and the block combines the 64 values with a **tree reduction** (`cuda.syncthreads()` between steps).
- Thread 0 of each block calls `cuda.atomic.max(out, 0, block_max)` on a 1-element `float32` device array initialised to `-inf`.
- Return the maximum as a Python `float`.

## Inputs
- `x`: `float32`, shape `(n,)` with `n >= 1`. No NaNs; `-inf` and `+inf` may appear.
- `n` may be smaller than the 512 threads of the grid: threads with no elements must contribute the **identity** of max, which is `-inf`.

Max is exact (no rounding), so the result is identical whatever order the atomics arrive in.

## Runtime
GPU in Colab; checked here on Numba's CUDA simulator.

**Runtime → Change runtime type → GPU** before running.

Write your solution in the next cell, then run the test cell.

In [ ]:
!pip -q install "numba-cuda[cu12]"

In [ ]:
import math
import numpy as np
from numba import cuda, float32

BLOCKS, TPB = 8, 64

def array_max(x):
    """x: float32 (n,), n >= 1. Returns max(x) as a Python float."""
    d_x = cuda.to_device(x)
    d_out = cuda.to_device(np.zeros(1, dtype=np.float32))   # TODO: is 0 the right starting value?
    max_kernel[BLOCKS, TPB](d_x, d_out)
    return float(d_out.copy_to_host()[0])

@cuda.jit
def max_kernel(x, out):
    i = cuda.grid(1)
    if i < x.shape[0]:
        cuda.atomic.max(out, 0, x[i])   # TODO: one atomic per element, and elements past 512 are skipped


In [ ]:
# Tests (same harness and cases as the website)
_HARNESS = '"""Test harness shared by build.py (CPython) and the browser runner (Pyodide).\n\nOne file, so a test that passes at build time is graded the same way in the browser.\nValues cross JSON with small tags: ndarray, tuple, DataFrame/Series, non-finite floats.\n"""\nimport json, math\n\ntry:\n    import numpy as np\nexcept ImportError:  # pure-python problems never need it\n    np = None\n\n\ndef enc(x):\n    if np is not None and isinstance(x, np.ndarray):\n        return {\'__nd__\': enc(x.tolist()), \'dtype\': str(x.dtype)}\n    if np is not None and isinstance(x, np.generic):\n        return enc(x.item())\n    if isinstance(x, float) and not math.isfinite(x):\n        return {\'__f__\': repr(x)}\n    if isinstance(x, tuple):\n        return {\'__tup__\': [enc(v) for v in x]}\n    if isinstance(x, list):\n        return [enc(v) for v in x]\n    if isinstance(x, dict):\n        return {\'__dict__\': [[enc(k), enc(v)] for k, v in x.items()]}\n    t = type(x).__name__\n    if t == \'Tensor\':  # torch, in Colab and at build time\n        return enc(x.detach().cpu().numpy())\n    if t == \'DataFrame\':\n        # Cells go through enc() as Python values, so floats keep full precision (to_json caps at 15 digits).\n        return {\'__df__\': {\'columns\': [enc(c) for c in x.columns], \'data\': [enc(list(r)) for r in x.itertuples(index=False, name=None)]},\n                \'dtypes\': [str(d) for d in x.dtypes]}\n    if t == \'Series\':\n        return {\'__sr__\': enc(x.tolist()), \'name\': x.name}\n    return x\n\n\ndef dec(x):\n    if isinstance(x, list):\n        return [dec(v) for v in x]\n    if not isinstance(x, dict):\n        return x\n    if \'__nd__\' in x:\n        return np.array(dec(x[\'__nd__\']), dtype=x[\'dtype\'])\n    if \'__f__\' in x:\n        return float(x[\'__f__\'])\n    if \'__tup__\' in x:\n        return tuple(dec(v) for v in x[\'__tup__\'])\n    if \'__dict__\' in x:\n        return {_hashable(dec(k)): dec(v) for k, v in x[\'__dict__\']}\n    if \'__df__\' in x:\n        import pandas as pd\n        s = x[\'__df__\']\n        df = pd.DataFrame([dec(r) for r in s[\'data\']], columns=[dec(c) for c in s[\'columns\']])\n        for c, d in zip(df.columns, x[\'dtypes\']):\n            if d.startswith((\'int\', \'float\', \'bool\')):\n                df[c] = df[c].astype(d)\n        return df\n    if \'__sr__\' in x:\n        import pandas as pd\n        return pd.Series(dec(x[\'__sr__\']), name=x[\'name\'])\n    return x\n\n\ndef _hashable(k):\n    return tuple(k) if isinstance(k, list) else k\n\n\ndef same(got, exp, rtol=1e-6, atol=1e-8, path=\'output\'):\n    """(ok, why). Floats compare with tolerance, everything else exactly. Shapes must match."""\n    if type(exp).__name__ == \'DataFrame\':\n        if type(got).__name__ != \'DataFrame\':\n            return False, f\'{path}: expected a DataFrame, got {type(got).__name__}\'\n        if list(got.columns) != list(exp.columns):\n            return False, f\'{path}: columns {list(got.columns)} != expected {list(exp.columns)}\'\n        if got.shape != exp.shape:\n            return False, f\'{path}: shape {got.shape} != expected {exp.shape}\'\n        for c in exp.columns:\n            ok, why = same(got[c].tolist(), exp[c].tolist(), rtol, atol, f\'{path}[{c!r}]\')\n            if not ok:\n                return ok, why\n        return True, \'\'\n    if type(exp).__name__ == \'Series\':\n        exp = exp.tolist()\n        got = got.tolist() if type(got).__name__ == \'Series\' else got\n    if np is not None and isinstance(exp, np.ndarray):\n        try:\n            g = np.asarray(got)\n        except Exception:\n            return False, f\'{path}: expected an array, got {type(got).__name__}\'\n        if g.shape != exp.shape:\n            return False, f\'{path}: shape {g.shape} != expected {exp.shape}\'\n        if exp.dtype.kind in \'fc\' or g.dtype.kind in \'fc\':\n            if g.dtype.kind not in \'fciub\':\n                return False, f\'{path}: expected numbers, got dtype {g.dtype}\'\n            bad = ~np.isclose(g.astype(complex if exp.dtype.kind == \'c\' else float), exp, rtol=rtol, atol=atol, equal_nan=True)\n            if bad.any():\n                i = tuple(int(v) for v in np.argwhere(bad)[0])\n                return False, f\'{path}{list(i)}: {g[i].tolist()!r} != expected {exp[i].tolist()!r}\'\n            return True, \'\'\n        if not np.array_equal(g, exp):\n            i = tuple(int(v) for v in np.argwhere(g != exp)[0]) if g.shape else ()\n            return False, f\'{path}{list(i)}: {g[i].tolist()!r} != expected {exp[i].tolist()!r}\'\n        return True, \'\'\n    if isinstance(exp, float) or (isinstance(exp, int) and isinstance(got, float) and not isinstance(exp, bool)):\n        if np is not None and isinstance(got, np.generic):\n            got = got.item()\n        if not isinstance(got, (int, float)) or isinstance(got, bool):\n            return False, f\'{path}: expected a number, got {type(got).__name__}\'\n        if math.isnan(exp) and math.isnan(got):\n            return True, \'\'\n        return (True, \'\') if math.isclose(got, exp, rel_tol=rtol, abs_tol=atol) else (False, f\'{path}: {got!r} != expected {exp!r}\')\n    if isinstance(exp, (list, tuple)):\n        if np is not None and isinstance(got, np.ndarray):\n            got = got.tolist()\n        if not isinstance(got, (list, tuple)):\n            return False, f\'{path}: expected a {type(exp).__name__}, got {type(got).__name__}\'\n        if len(got) != len(exp):\n            return False, f\'{path}: length {len(got)} != expected {len(exp)}\'\n        for i, (g, e) in enumerate(zip(got, exp)):\n            ok, why = same(g, e, rtol, atol, f\'{path}[{i}]\')\n            if not ok:\n                return ok, why\n        return True, \'\'\n    if isinstance(exp, dict):\n        if not isinstance(got, dict):\n            return False, f\'{path}: expected a dict, got {type(got).__name__}\'\n        if set(got) != set(exp):\n            return False, f\'{path}: keys {sorted(map(str, got))} != expected {sorted(map(str, exp))}\'\n        for k in exp:\n            ok, why = same(got[k], exp[k], rtol, atol, f\'{path}[{k!r}]\')\n            if not ok:\n                return ok, why\n        return True, \'\'\n    if np is not None and isinstance(got, np.generic):\n        got = got.item()\n    return (True, \'\') if got == exp else (False, f\'{path}: {got!r} != expected {exp!r}\')\n\n\ndef run_sql(query, setup):\n    """Run `setup` (CREATE/INSERT script) in a fresh in-memory db, then `query`; rows as lists."""\n    import sqlite3  # a separate package in Pyodide: only SQL problems load it\n    con = sqlite3.connect(\':memory:\')\n    try:\n        con.executescript(setup)\n        cur = con.execute(query)\n        return [[c[0] for c in cur.description], [list(r) for r in cur.fetchall()]]\n    finally:\n        con.close()\n\n\ndef show_args(args, runtime, n=600):\n    """Arguments as the learner would write them: one per line, SQL setup as-is."""\n    if runtime == \'sql\':\n        return args[0] if len(args[0]) <= n else args[0][:n] + \' …\'\n    return \'\\n\'.join(short(a, n) for a in args)\n\n\ndef short(x, n=300):\n    s = repr(x)\n    return s if len(s) <= n else s[:n] + \' …\'\n\n\ndef grade(fn, tests, runtime, rtol=1e-6, atol=1e-8):\n    """Run `fn` (a callable, or SQL text for runtime \'sql\') on each test; list of result dicts."""\n    out = []\n    for t in tests:\n        args, exp = prep(dec(t[\'a\']), runtime), dec(t[\'e\'])\n        try:\n            got = run_sql(fn, args[0]) if runtime == \'sql\' else plain(fn(*_copy(args)))\n            ok, why = same(got, exp, rtol, atol)\n        except Exception as e:  # learner code raised: report it like a failed test\n            got, ok, why = None, False, f\'{type(e).__name__}: {e}\'\n        out.append({\'ok\': ok, \'why\': why, \'got\': short(got) if not ok else \'\',\n                    \'inp\': show_args(args, runtime, 400) if not ok else \'\'})\n    return out\n\n\ndef prep(args, runtime):\n    """Arrays cross JSON as ndarrays; torch problems take tensors."""\n    if runtime == \'torch\':\n        import torch\n        return [torch.as_tensor(a) if np is not None and isinstance(a, np.ndarray) else a for a in args]\n    if runtime == \'triton\':                                  # Colab only: kernels need CUDA tensors\n        import torch\n        return [torch.as_tensor(a).cuda() if np is not None and isinstance(a, np.ndarray) else a for a in args]\n    return args\n\n\ndef plain(x):\n    """Framework arrays (torch, jax) -> numpy, recursively, so results compare and encode the same way."""\n    t = type(x).__name__\n    if t == \'Tensor\':\n        return x.detach().cpu().numpy()\n    if t in (\'ArrayImpl\', \'Array\') and hasattr(x, \'__array__\'):\n        return np.asarray(x)\n    if isinstance(x, tuple):\n        return tuple(plain(v) for v in x)\n    if isinstance(x, list):\n        return [plain(v) for v in x]\n    if isinstance(x, dict):\n        return {k: plain(v) for k, v in x.items()}\n    return x\n\n\ndef _copy(args):\n    # Tests reuse inputs across Run and Submit, so a function that mutates its input mustn\'t poison later cases.\n    import copy\n    return copy.deepcopy(args)\n'
_TESTS = '[{"a": [{"__nd__": [3.0, -1.0, 7.5, 2.0], "dtype": "float32"}], "e": 7.5}, {"a": [{"__nd__": [-5.0, -2.0, -9.0], "dtype": "float32"}], "e": -2.0}, {"a": [{"__nd__": [42.0], "dtype": "float32"}], "e": 42.0}, {"a": [{"__nd__": [{"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}], "dtype": "float32"}], "e": {"__f__": "-inf"}}, {"a": [{"__nd__": [0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, {"__f__": "inf"}], "dtype": "float32"}], "e": {"__f__": "inf"}}, {"a": [{"__nd__": [-0.0, -1.0, -2.0, -3.0, -4.0, -5.0, -6.0, -7.0, -8.0, -9.0, -10.0, -11.0, -12.0, -13.0, -14.0, -15.0, -16.0, -17.0, -18.0, -19.0, -20.0, -21.0, -22.0, -23.0, -24.0, -25.0, -26.0, -27.0, -28.0, -29.0, -30.0, -31.0, -32.0, -33.0, -34.0, -35.0, -36.0, -37.0, -38.0, -39.0, -40.0, -41.0, -42.0, -43.0, -44.0, -45.0, -46.0, -47.0, -48.0, -49.0, -50.0, -51.0, -52.0, -53.0, -54.0, -55.0, -56.0, -57.0, -58.0, -59.0, -60.0, -61.0, -62.0, -63.0, -64.0, -65.0, -66.0, -67.0, -68.0, -69.0, -70.0, -71.0, -72.0, -73.0, -74.0, -75.0, -76.0, -77.0, -78.0, -79.0, -80.0, -81.0, -82.0, -83.0, -84.0, -85.0, -86.0, -87.0, -88.0, -89.0, -90.0, -91.0, -92.0, -93.0, -94.0, -95.0, -96.0, -97.0, -98.0, -99.0, -100.0, -101.0, -102.0, -103.0, -104.0, -105.0, -106.0, -107.0, -108.0, -109.0, -110.0, -111.0, -112.0, -113.0, -114.0, -115.0, -116.0, -117.0, -118.0, -119.0, -120.0, -121.0, -122.0, -123.0, -124.0, -125.0, -126.0, -127.0, -128.0, -129.0, -130.0, -131.0, -132.0, -133.0, -134.0, -135.0, -136.0, -137.0, -138.0, -139.0, -140.0, -141.0, -142.0, -143.0, -144.0, -145.0, -146.0, -147.0, -148.0, -149.0, -150.0, -151.0, -152.0, -153.0, -154.0, -155.0, -156.0, -157.0, -158.0, -159.0, -160.0, -161.0, -162.0, -163.0, -164.0, -165.0, -166.0, -167.0, -168.0, -169.0, -170.0, -171.0, -172.0, -173.0, -174.0, -175.0, -176.0, -177.0, -178.0, -179.0, -180.0, -181.0, -182.0, -183.0, -184.0, -185.0, -186.0, -187.0, -188.0, -189.0, -190.0, -191.0, -192.0, -193.0, -194.0, -195.0, -196.0, -197.0, -198.0, -199.0, -200.0, -201.0, -202.0, -203.0, -204.0, -205.0, -206.0, -207.0, -208.0, -209.0, -210.0, -211.0, -212.0, -213.0, -214.0, -215.0, -216.0, -217.0, -218.0, -219.0, -220.0, -221.0, -222.0, -223.0, -224.0, -225.0, -226.0, -227.0, -228.0, -229.0, -230.0, -231.0, -232.0, -233.0, -234.0, -235.0, -236.0, -237.0, -238.0, -239.0, -240.0, -241.0, -242.0, -243.0, -244.0, -245.0, -246.0, -247.0, -248.0, -249.0, -250.0, -251.0, -252.0, -253.0, -254.0, -255.0, -256.0, -257.0, -258.0, -259.0, -260.0, -261.0, -262.0, -263.0, -264.0, -265.0, -266.0, -267.0, -268.0, -269.0, -270.0, -271.0, -272.0, -273.0, -274.0, -275.0, -276.0, -277.0, -278.0, -279.0, -280.0, -281.0, -282.0, -283.0, -284.0, -285.0, -286.0, -287.0, -288.0, -289.0, -290.0, -291.0, -292.0, -293.0, -294.0, -295.0, -296.0, -297.0, -298.0, -299.0, -300.0, -301.0, -302.0, -303.0, -304.0, -305.0, -306.0, -307.0, -308.0, -309.0, -310.0, -311.0, -312.0, -313.0, -314.0, -315.0, -316.0, -317.0, -318.0, -319.0, -320.0, -321.0, -322.0, -323.0, -324.0, -325.0, -326.0, -327.0, -328.0, -329.0, -330.0, -331.0, -332.0, -333.0, -334.0, -335.0, -336.0, -337.0, -338.0, -339.0, -340.0, -341.0, -342.0, -343.0, -344.0, -345.0, -346.0, -347.0, -348.0, -349.0, -350.0, -351.0, -352.0, -353.0, -354.0, -355.0, -356.0, -357.0, -358.0, -359.0, -360.0, -361.0, -362.0, -363.0, -364.0, -365.0, -366.0, -367.0, -368.0, -369.0, -370.0, -371.0, -372.0, -373.0, -374.0, -375.0, -376.0, -377.0, -378.0, -379.0, -380.0, -381.0, -382.0, -383.0, -384.0, -385.0, -386.0, -387.0, -388.0, -389.0, -390.0, -391.0, -392.0, -393.0, -394.0, -395.0, -396.0, -397.0, -398.0, -399.0, -400.0, -401.0, -402.0, -403.0, -404.0, -405.0, -406.0, -407.0, -408.0, -409.0, -410.0, -411.0, -412.0, -413.0, -414.0, -415.0, -416.0, -417.0, -418.0, -419.0, -420.0, -421.0, -422.0, -423.0, -424.0, -425.0, -426.0, -427.0, -428.0, -429.0, -430.0, -431.0, -432.0, -433.0, -434.0, -435.0, -436.0, -437.0, -438.0, -439.0, -440.0, -441.0, -442.0, -443.0, -444.0, -445.0, -446.0, -447.0, -448.0, -449.0, -450.0, -451.0, -452.0, -453.0, -454.0, -455.0, -456.0, -457.0, -458.0, -459.0, -460.0, -461.0, -462.0, -463.0, -464.0, -465.0, -466.0, -467.0, -468.0, -469.0, -470.0, -471.0, -472.0, -473.0, -474.0, -475.0, -476.0, -477.0, -478.0, -479.0, -480.0, -481.0, -482.0, -483.0, -484.0, -485.0, -486.0, -487.0, -488.0, -489.0, -490.0, -491.0, -492.0, -493.0, -494.0, -495.0, -496.0, -497.0, -498.0, -499.0, -500.0, -501.0, -502.0, -503.0, -504.0, -505.0, -506.0, -507.0, -508.0, -509.0, -510.0, -511.0, -512.0, -513.0, -514.0, -515.0, -516.0, -517.0, -518.0, -519.0, -520.0, -521.0, -522.0, -523.0, -524.0, -525.0, -526.0, -527.0, -528.0, -529.0, -530.0, -531.0, -532.0, -533.0, -534.0, -535.0, -536.0, -537.0, -538.0, -539.0, -540.0, -541.0, -542.0, -543.0, -544.0, -545.0, -546.0, -547.0, -548.0, -549.0, -550.0, -551.0, -552.0, -553.0, -554.0, -555.0, -556.0, -557.0, -558.0, -559.0, -560.0, -561.0, -562.0, -563.0, -564.0, -565.0, -566.0, -567.0, -568.0, -569.0, -570.0, -571.0, -572.0, -573.0, -574.0, -575.0, -576.0, -577.0, -578.0, -579.0, -580.0, -581.0, -582.0, -583.0, -584.0, -585.0, -586.0, -587.0, -588.0, -589.0, -590.0, -591.0, -592.0, -593.0, -594.0, -595.0, -596.0, -597.0, -598.0, -599.0, -600.0, -601.0, -602.0, -603.0, -604.0, -605.0, -606.0, -607.0, -608.0, -609.0, -610.0, -611.0, -612.0, -613.0, -614.0, -615.0, -616.0, -617.0, -618.0, -619.0, -620.0, -621.0, -622.0, -623.0, -624.0, -625.0, -626.0, -627.0, -628.0, -629.0, -630.0, -631.0, -632.0, -633.0, -634.0, -635.0, -636.0, -637.0, -638.0, -639.0, -640.0, -641.0, -642.0, -643.0, -644.0, -645.0, -646.0, -647.0, -648.0, -649.0, -650.0, -651.0, -652.0, -653.0, -654.0, -655.0, -656.0, -657.0, -658.0, -659.0, -660.0, -661.0, -662.0, -663.0, -664.0, -665.0, -666.0, -667.0, -668.0, -669.0, -670.0, -671.0, -672.0, -673.0, -674.0, -675.0, -676.0, -677.0, -678.0, -679.0, -680.0, -681.0, -682.0, -683.0, -684.0, -685.0, -686.0, -687.0, -688.0, -689.0, -690.0, -691.0, -692.0, -693.0, -694.0, -695.0, -696.0, -697.0, -698.0, -699.0, -700.0, -701.0, -702.0, -703.0, -704.0, -705.0, -706.0, -707.0, -708.0, -709.0, -710.0, -711.0, -712.0, -713.0, -714.0, -715.0, -716.0, -717.0, -718.0, -719.0, -720.0, -721.0, -722.0, -723.0, -724.0, -725.0, -726.0, -727.0, -728.0, -729.0, -730.0, -731.0, -732.0, -733.0, -734.0, -735.0, -736.0, -737.0, -738.0, -739.0, -740.0, -741.0, -742.0, -743.0, -744.0, -745.0, -746.0, -747.0, -748.0, -749.0, -750.0, -751.0, -752.0, -753.0, -754.0, -755.0, -756.0, -757.0, -758.0, -759.0, -760.0, -761.0, -762.0, -763.0, -764.0, -765.0, -766.0, -767.0, -768.0, -769.0, -770.0, -771.0, -772.0, -773.0, -774.0, -775.0, -776.0, -777.0, -778.0, -779.0, -780.0, -781.0, -782.0, -783.0, -784.0, -785.0, -786.0, -787.0, -788.0, -789.0, -790.0, -791.0, -792.0, -793.0, -794.0, -795.0, -796.0, -797.0, -798.0, -799.0, -800.0, -801.0, -802.0, -803.0, -804.0, -805.0, -806.0, -807.0, -808.0, -809.0, -810.0, -811.0, -812.0, -813.0, -814.0, -815.0, -816.0, -817.0, -818.0, -819.0, -820.0, -821.0, -822.0, -823.0, -824.0, -825.0, -826.0, -827.0, -828.0, -829.0, -830.0, -831.0, -832.0, -833.0, -834.0, -835.0, -836.0, -837.0, -838.0, -839.0, -840.0, -841.0, -842.0, -843.0, -844.0, -845.0, -846.0, -847.0, -848.0, -849.0, -850.0, -851.0, -852.0, -853.0, -854.0, -855.0, -856.0, -857.0, -858.0, -859.0, -860.0, -861.0, -862.0, -863.0, -864.0, -865.0, -866.0, -867.0, -868.0, -869.0, -870.0, -871.0, -872.0, -873.0, -874.0, -875.0, -876.0, -877.0, -878.0, -879.0, -880.0, -881.0, -882.0, -883.0, -884.0, -885.0, -886.0, -887.0, -888.0, -889.0, -890.0, -891.0, -892.0, -893.0, -894.0, -895.0, -896.0, -897.0, -898.0, -899.0, -900.0, -901.0, -902.0, -903.0, -904.0, -905.0, -906.0, -907.0, -908.0, -909.0, -910.0, -911.0, -912.0, -913.0, -914.0, -915.0, -916.0, -917.0, -918.0, -919.0, -920.0, -921.0, -922.0, -923.0, -924.0, -925.0, -926.0, -927.0, -928.0, -929.0, -930.0, -931.0, -932.0, -933.0, -934.0, -935.0, -936.0, -937.0, -938.0, -939.0, -940.0, -941.0, -942.0, -943.0, -944.0, -945.0, -946.0, -947.0, -948.0, -949.0, -950.0, -951.0, -952.0, -953.0, -954.0, -955.0, -956.0, -957.0, -958.0, -959.0, -960.0, -961.0, -962.0, -963.0, -964.0, -965.0, -966.0, -967.0, -968.0, -969.0, -970.0, -971.0, -972.0, -973.0, -974.0, -975.0, -976.0, -977.0, -978.0, -979.0, -980.0, -981.0, -982.0, -983.0, -984.0, -985.0, -986.0, -987.0, -988.0, -989.0, -990.0, -991.0, -992.0, -993.0, -994.0, -995.0, -996.0, -997.0, -998.0, -999.0, -1000.0, -1001.0, -1002.0, -1003.0, -1004.0, -1005.0, -1006.0, -1007.0, -1008.0, -1009.0, -1010.0, -1011.0, -1012.0, -1013.0, -1014.0, -1015.0, -1016.0, -1017.0, -1018.0, -1019.0, -1020.0, -1021.0, -1022.0, -1023.0, -1024.0, -1025.0, -1026.0, -1027.0, -1028.0, -1029.0, -1030.0, -1031.0, -1032.0, -1033.0, -1034.0, -1035.0, -1036.0, -1037.0, -1038.0, -1039.0, -1040.0, -1041.0, -1042.0, -1043.0, -1044.0, -1045.0, -1046.0, -1047.0, -1048.0, -1049.0, -1050.0, -1051.0, -1052.0, -1053.0, -1054.0, -1055.0, -1056.0, -1057.0, -1058.0, -1059.0, -1060.0, -1061.0, -1062.0, -1063.0, -1064.0, -1065.0, -1066.0, -1067.0, -1068.0, -1069.0, -1070.0, -1071.0, -1072.0, -1073.0, -1074.0, -1075.0, -1076.0, -1077.0, -1078.0, -1079.0, -1080.0, -1081.0, -1082.0, -1083.0, -1084.0, -1085.0, -1086.0, -1087.0, -1088.0, -1089.0, -1090.0, -1091.0, -1092.0, -1093.0, -1094.0, -1095.0, -1096.0, -1097.0, -1098.0, -1099.0, -1100.0, -1101.0, -1102.0, -1103.0, -1104.0, -1105.0, -1106.0, -1107.0, -1108.0, -1109.0, -1110.0, -1111.0, -1112.0, -1113.0, -1114.0, -1115.0, -1116.0, -1117.0, -1118.0, -1119.0, -1120.0, -1121.0, -1122.0, -1123.0, -1124.0, -1125.0, -1126.0, -1127.0, -1128.0, -1129.0, -1130.0, -1131.0, -1132.0, -1133.0, -1134.0, -1135.0, -1136.0, -1137.0, -1138.0, -1139.0, -1140.0, -1141.0, -1142.0, -1143.0, -1144.0, -1145.0, -1146.0, -1147.0, -1148.0, -1149.0, -1150.0, -1151.0, -1152.0, -1153.0, -1154.0, -1155.0, -1156.0, -1157.0, -1158.0, -1159.0, -1160.0, -1161.0, -1162.0, -1163.0, -1164.0, -1165.0, -1166.0, -1167.0, -1168.0, -1169.0, -1170.0, -1171.0, -1172.0, -1173.0, -1174.0, -1175.0, -1176.0, -1177.0, -1178.0, -1179.0, -1180.0, -1181.0, -1182.0, -1183.0, -1184.0, -1185.0, -1186.0, -1187.0, -1188.0, -1189.0, -1190.0, -1191.0, -1192.0, -1193.0, -1194.0, -1195.0, -1196.0, -1197.0, -1198.0, -1199.0, -1200.0, -1201.0, -1202.0, -1203.0, -1204.0, -1205.0, -1206.0, -1207.0, -1208.0, -1209.0, -1210.0, -1211.0, -1212.0, -1213.0, -1214.0, -1215.0, -1216.0, -1217.0, -1218.0, -1219.0, -1220.0, -1221.0, -1222.0, -1223.0, -1224.0, -1225.0, -1226.0, -1227.0, -1228.0, -1229.0, -1230.0, -1231.0, -1232.0, -1233.0, -1234.0, -1235.0, -1236.0, -1237.0, -1238.0, -1239.0, -1240.0, -1241.0, -1242.0, -1243.0, -1244.0, -1245.0, -1246.0, -1247.0, -1248.0, -1249.0, -1250.0, -1251.0, -1252.0, -1253.0, -1254.0, -1255.0, -1256.0, -1257.0, -1258.0, -1259.0, -1260.0, -1261.0, -1262.0, -1263.0, -1264.0, -1265.0, -1266.0, -1267.0, -1268.0, -1269.0, -1270.0, -1271.0, -1272.0, -1273.0, -1274.0, -1275.0, -1276.0, -1277.0, -1278.0, -1279.0, -1280.0, -1281.0, -1282.0, -1283.0, -1284.0, -1285.0, -1286.0, -1287.0, -1288.0, -1289.0, -1290.0, -1291.0, -1292.0, -1293.0, -1294.0, -1295.0, -1296.0, -1297.0, -1298.0, -1299.0, -1300.0, -1301.0, -1302.0, -1303.0, -1304.0, -1305.0, -1306.0, -1307.0, -1308.0, -1309.0, -1310.0, -1311.0, -1312.0, -1313.0, -1314.0, -1315.0, -1316.0, -1317.0, -1318.0, -1319.0, -1320.0, -1321.0, -1322.0, -1323.0, -1324.0, -1325.0, -1326.0, -1327.0, -1328.0, -1329.0, -1330.0, -1331.0, -1332.0, -1333.0, -1334.0, -1335.0, -1336.0, -1337.0, -1338.0, -1339.0, -1340.0, -1341.0, -1342.0, -1343.0, -1344.0, -1345.0, -1346.0, -1347.0, -1348.0, -1349.0, -1350.0, -1351.0, -1352.0, -1353.0, -1354.0, -1355.0, -1356.0, -1357.0, -1358.0, -1359.0, -1360.0, -1361.0, -1362.0, -1363.0, -1364.0, -1365.0, -1366.0, -1367.0, -1368.0, -1369.0, -1370.0, -1371.0, -1372.0, -1373.0, -1374.0, -1375.0, -1376.0, -1377.0, -1378.0, -1379.0, -1380.0, -1381.0, -1382.0, -1383.0, -1384.0, -1385.0, -1386.0, -1387.0, -1388.0, -1389.0, -1390.0, -1391.0, -1392.0, -1393.0, -1394.0, -1395.0, -1396.0, -1397.0, -1398.0, -1399.0, -1400.0, -1401.0, -1402.0, -1403.0, -1404.0, -1405.0, -1406.0, -1407.0, -1408.0, -1409.0, -1410.0, -1411.0, -1412.0, -1413.0, -1414.0, -1415.0, -1416.0, -1417.0, -1418.0, -1419.0, -1420.0, -1421.0, -1422.0, -1423.0, -1424.0, -1425.0, -1426.0, -1427.0, -1428.0, -1429.0, -1430.0, -1431.0, -1432.0, -1433.0, -1434.0, -1435.0, -1436.0, -1437.0, -1438.0, -1439.0, -1440.0, -1441.0, -1442.0, -1443.0, -1444.0, -1445.0, -1446.0, -1447.0, -1448.0, -1449.0, -1450.0, -1451.0, -1452.0, -1453.0, -1454.0, -1455.0, -1456.0, -1457.0, -1458.0, -1459.0, -1460.0, -1461.0, -1462.0, -1463.0, -1464.0, -1465.0, -1466.0, -1467.0, -1468.0, -1469.0, -1470.0, -1471.0, -1472.0, -1473.0, -1474.0, -1475.0, -1476.0, -1477.0, -1478.0, -1479.0, -1480.0, -1481.0, -1482.0, -1483.0, -1484.0, -1485.0, -1486.0, -1487.0, -1488.0, -1489.0, -1490.0, -1491.0, -1492.0, -1493.0, -1494.0, -1495.0, -1496.0, -1497.0, -1498.0, -1499.0, -1500.0, -1501.0, -1502.0, -1503.0, -1504.0, -1505.0, -1506.0, -1507.0, -1508.0, -1509.0, -1510.0, -1511.0, -1512.0, -1513.0, -1514.0, -1515.0, -1516.0, -1517.0, -1518.0, -1519.0, -1520.0, -1521.0, -1522.0, -1523.0, -1524.0, -1525.0, -1526.0, -1527.0, -1528.0, -1529.0, -1530.0, -1531.0, -1532.0, -1533.0, -1534.0, -1535.0, -1536.0, -1537.0, -1538.0, -1539.0, -1540.0, -1541.0, -1542.0, -1543.0, -1544.0, -1545.0, -1546.0, -1547.0, -1548.0, -1549.0, -1550.0, -1551.0, -1552.0, -1553.0, -1554.0, -1555.0, -1556.0, -1557.0, -1558.0, -1559.0, -1560.0, -1561.0, -1562.0, -1563.0, -1564.0, -1565.0, -1566.0, -1567.0, -1568.0, -1569.0, -1570.0, -1571.0, -1572.0, -1573.0, -1574.0, -1575.0, -1576.0, -1577.0, -1578.0, -1579.0, -1580.0, -1581.0, -1582.0, -1583.0, -1584.0, -1585.0, -1586.0, -1587.0, -1588.0, -1589.0, -1590.0, -1591.0, -1592.0, -1593.0, -1594.0, -1595.0, -1596.0, -1597.0, -1598.0, -1599.0, -1600.0, -1601.0, -1602.0, -1603.0, -1604.0, -1605.0, -1606.0, -1607.0, -1608.0, -1609.0, -1610.0, -1611.0, -1612.0, -1613.0, -1614.0, -1615.0, -1616.0, -1617.0, -1618.0, -1619.0, -1620.0, -1621.0, -1622.0, -1623.0, -1624.0, -1625.0, -1626.0, -1627.0, -1628.0, -1629.0, -1630.0, -1631.0, -1632.0, -1633.0, -1634.0, -1635.0, -1636.0, -1637.0, -1638.0, -1639.0, -1640.0, -1641.0, -1642.0, -1643.0, -1644.0, -1645.0, -1646.0, -1647.0, -1648.0, -1649.0, -1650.0, -1651.0, -1652.0, -1653.0, -1654.0, -1655.0, -1656.0, -1657.0, -1658.0, -1659.0, -1660.0, -1661.0, -1662.0, -1663.0, -1664.0, -1665.0, -1666.0, -1667.0, -1668.0, -1669.0, -1670.0, -1671.0, -1672.0, -1673.0, -1674.0, -1675.0, -1676.0, -1677.0, -1678.0, -1679.0, -1680.0, -1681.0, -1682.0, -1683.0, -1684.0, -1685.0, -1686.0, -1687.0, -1688.0, -1689.0, -1690.0, -1691.0, -1692.0, -1693.0, -1694.0, -1695.0, -1696.0, -1697.0, -1698.0, -1699.0, -1700.0, -1701.0, -1702.0, -1703.0, -1704.0, -1705.0, -1706.0, -1707.0, -1708.0, -1709.0, -1710.0, -1711.0, -1712.0, -1713.0, -1714.0, -1715.0, -1716.0, -1717.0, -1718.0, -1719.0, -1720.0, -1721.0, -1722.0, -1723.0, -1724.0, -1725.0, -1726.0, -1727.0, -1728.0, -1729.0, -1730.0, -1731.0, -1732.0, -1733.0, -1734.0, -1735.0, -1736.0, -1737.0, -1738.0, -1739.0, -1740.0, -1741.0, -1742.0, -1743.0, -1744.0, -1745.0, -1746.0, -1747.0, -1748.0, -1749.0, -1750.0, -1751.0, -1752.0, -1753.0, -1754.0, -1755.0, -1756.0, -1757.0, -1758.0, -1759.0, -1760.0, -1761.0, -1762.0, -1763.0, -1764.0, -1765.0, -1766.0, -1767.0, -1768.0, -1769.0, -1770.0, -1771.0, -1772.0, -1773.0, -1774.0, -1775.0, -1776.0, -1777.0, -1778.0, -1779.0, -1780.0, -1781.0, -1782.0, -1783.0, -1784.0, -1785.0, -1786.0, -1787.0, -1788.0, -1789.0, -1790.0, -1791.0, -1792.0, -1793.0, -1794.0, -1795.0, -1796.0, -1797.0, -1798.0, -1799.0, -1800.0, -1801.0, -1802.0, -1803.0, -1804.0, -1805.0, -1806.0, -1807.0, -1808.0, -1809.0, -1810.0, -1811.0, -1812.0, -1813.0, -1814.0, -1815.0, -1816.0, -1817.0, -1818.0, -1819.0, -1820.0, -1821.0, -1822.0, -1823.0, -1824.0, -1825.0, -1826.0, -1827.0, -1828.0, -1829.0, -1830.0, -1831.0, -1832.0, -1833.0, -1834.0, -1835.0, -1836.0, -1837.0, -1838.0, -1839.0, -1840.0, -1841.0, -1842.0, -1843.0, -1844.0, -1845.0, -1846.0, -1847.0, -1848.0, -1849.0, -1850.0, -1851.0, -1852.0, -1853.0, -1854.0, -1855.0, -1856.0, -1857.0, -1858.0, -1859.0, -1860.0, -1861.0, -1862.0, -1863.0, -1864.0, -1865.0, -1866.0, -1867.0, -1868.0, -1869.0, -1870.0, -1871.0, -1872.0, -1873.0, -1874.0, -1875.0, -1876.0, -1877.0, -1878.0, -1879.0, -1880.0, -1881.0, -1882.0, -1883.0, -1884.0, -1885.0, -1886.0, -1887.0, -1888.0, -1889.0, -1890.0, -1891.0, -1892.0, -1893.0, -1894.0, -1895.0, -1896.0, -1897.0, -1898.0, -1899.0, -1900.0, -1901.0, -1902.0, -1903.0, -1904.0, -1905.0, -1906.0, -1907.0, -1908.0, -1909.0, -1910.0, -1911.0, -1912.0, -1913.0, -1914.0, -1915.0, -1916.0, -1917.0, -1918.0, -1919.0, -1920.0, -1921.0, -1922.0, -1923.0, -1924.0, -1925.0, -1926.0, -1927.0, -1928.0, -1929.0, -1930.0, -1931.0, -1932.0, -1933.0, -1934.0, -1935.0, -1936.0, -1937.0, -1938.0, -1939.0, -1940.0, -1941.0, -1942.0, -1943.0, -1944.0, -1945.0, -1946.0, -1947.0, -1948.0, -1949.0, -1950.0, -1951.0, -1952.0, -1953.0, -1954.0, -1955.0, -1956.0, -1957.0, -1958.0, -1959.0, -1960.0, -1961.0, -1962.0, -1963.0, -1964.0, -1965.0, -1966.0, -1967.0, -1968.0, -1969.0, -1970.0, -1971.0, -1972.0, -1973.0, -1974.0, -1975.0, -1976.0, -1977.0, -1978.0, -1979.0, -1980.0, -1981.0, -1982.0, -1983.0, -1984.0, -1985.0, -1986.0, -1987.0, -1988.0, -1989.0, -1990.0, -1991.0, -1992.0, -1993.0, -1994.0, -1995.0, -1996.0, -1997.0, -1998.0, -1999.0], "dtype": "float32"}], "e": -0.0}, {"a": [{"__nd__": [-0.39530128240585327, 0.26391488313674927, 0.6071282625198364, -0.9721598029136658, 0.7676642537117004, 0.2550581097602844, 0.7829798460006714, 0.27241823077201843, 1.1620081663131714, -0.937756359577179, 1.776099681854248, 1.2023898363113403, -0.5999578237533569, 0.660166323184967, 0.44476738572120667, -1.7457635402679443, 0.5952162742614746, -0.5853990912437439, -0.250013530254364, -0.6023864150047302, -0.42808401584625244, 0.0717347040772438, 0.09671912342309952, -1.5591518878936768, -0.2687060236930847, -1.344811201095581, -1.2707778215408325, -0.3469536006450653, 0.85530024766922, 0.6308565139770508, -0.6046580672264099, -0.7104208469390869, -0.8274099230766296, 0.14274021983146667, 0.9366500377655029, 0.018049856647849083, 0.6927666664123535, 0.3154090642929077, 1.504185676574707, -2.006690740585327, -2.1310341358184814, -0.1984395831823349, 0.6365395188331604, -0.4105492830276489, 0.42578068375587463, -1.1735447645187378, -1.259383201599121, -0.7165758609771729, 0.8413227200508118, 0.6963583827018738, -0.7132527232170105, 0.18111936748027802, -0.6475708484649658, 0.02005981095135212, -1.1142964363098145, -0.04655434936285019, -0.7357539534568787, -0.04435950890183449, -0.6520426273345947, -0.8217414617538452, 0.11343128979206085, -0.33779534697532654, 0.021671175956726074, -0.41908517479896545, 0.6137428879737854, -0.8156847357749939, 0.7687510251998901, 1.3319742679595947, -0.2688876986503601, 1.7452430725097656, 2.029698133468628, 0.4430750012397766, -1.3576050996780396, -1.6156939268112183, -0.17577868700027466, 1.7759298086166382, 0.5745498538017273, 0.758030891418457, -0.2509133219718933, -0.7441468834877014, 0.3719210922718048, -1.108778715133667, -0.19694438576698303, 0.5111522674560547, 0.03593415394425392, 0.5245642066001892, -0.445976585149765, 0.598179817199707, 0.40465039014816284, -1.649706244468689, -1.0422629117965698, -0.7234762907028198, 0.655586302280426, 0.3020801544189453, 0.2829882800579071, 0.172932431101799, 0.636476457118988, 0.9453478455543518, -0.554716169834137, -1.5909122228622437, 0.6673244833946228, -0.20942920446395874, 0.04880537837743759, -0.17380526661872864, -1.1125682592391968, -1.5106513500213623, 0.9642801880836487, 1.0930925607681274, 0.931239664554596, 0.20823465287685394, 0.2685207724571228, -0.41892966628074646, 0.06801223754882812, -0.4028360843658447, -0.5436861515045166, 0.20520316064357758, 1.6981004476547241, -1.883541464805603, 0.6015112996101379, 0.6756512522697449, -1.214406132698059, 0.3134499490261078, 0.10157496482133865, -0.4639827311038971, -0.4963243007659912, -1.4061732292175293, 1.707177758216858, 2.347567558288574, 1.5863759517669678, -0.28779324889183044, 0.28568199276924133, 0.3772081434726715, -0.8129329681396484, -0.25851336121559143, 0.5332453846931458, -0.4216229319572449, -1.759756088256836, -1.2680186033248901, 0.2549883723258972, -0.33706140518188477, -0.2982936203479767, 1.3559778928756714, 0.08478662371635437, -0.7174362540245056, -0.06496331840753555, 2.1579108238220215, -1.1264636516571045, 1.786948323249817, 1.4415949583053589, -0.16009530425071716, -0.33588939905166626, 1.4188083410263062, 0.24397017061710358, -0.5308330059051514, 1.8227617740631104, 0.17449237406253815, 0.3321458697319031, 0.03851839900016785, -0.3013107180595398, 0.12420020252466202, -0.3700959384441376, 0.5075311064720154, 1.055453896522522, 1.085198163986206, 1.1913931369781494, 1.4954686164855957, 0.7610350251197815, -0.6044836044311523, 0.9621555209159851, -1.6492067575454712, -0.7133713364601135, 0.28996866941452026, -1.6823936700820923, 1.4718807935714722, -1.0173481702804565, -1.4978022575378418, -0.7614302039146423, -0.7640095949172974, -0.026283925399184227, -0.15793462097644806, 1.031081199645996, -0.1664392650127411, 0.17555119097232819, -0.9899791479110718, 0.4972487688064575, -0.7749791145324707, -1.948292851448059, 1.3038350343704224, -1.056431770324707, -1.0897020101547241, -0.8766761422157288, 0.651117742061615, -1.5234259366989136, -0.39813995361328125, 0.5768841505050659, 1.8969395160675049, -0.28968507051467896, 0.05582650750875473, -0.8672504425048828, 1.981290578842163, -0.1405663788318634, -0.06256305426359177, 0.1467183381319046, -0.9161278009414673, -0.010194802656769753, -1.2612591981887817, 1.4888421297073364, -0.14947304129600525, -0.16486458480358124, 1.1681957244873047, -1.2158558368682861, 1.4012563228607178, 1.5222817659378052, -0.7692632079124451, 0.6907475590705872, -0.09757599979639053, 0.8827047944068909, 0.041959505528211594, -0.9001685976982117, 0.6281111240386963, -0.21847528219223022, 0.17924445867538452, 0.976440966129303, 1.215274691581726, 0.6291961669921875, -1.4211156368255615, 0.6944321990013123, -0.30050158500671387, 1.0722490549087524, -0.19584167003631592, 1.425998568534851, 0.5140710473060608, 6.793543434469029e-05, 0.29327264428138733, -1.7639750242233276, 0.5738072395324707, 0.11494091153144836, -1.0967977046966553, 0.4070016145706177, 0.3192577362060547, 0.35762062668800354, -0.11539345979690552, 0.7885177135467529, 0.8014461398124695, 0.09359147399663925, -0.06111520156264305, -0.4537959694862366, -0.595069944858551, 0.611762523651123, -0.5458992719650269, -1.412223219871521, 0.24595977365970612, 1.1074165105819702, -0.2736063599586487, 0.8682156205177307, 0.48149847984313965, -1.0081942081451416, 0.3224697709083557, -0.10889124125242233, 0.6822238564491272, -2.7856249809265137, 0.8312948346138, 0.7414124608039856, -0.9247534871101379, 1.0697388648986816, -0.15130722522735596, 0.18450219929218292, 0.8662132024765015, 1.3465386629104614, 0.173604816198349, 0.6882915496826172, 0.44626668095588684, -0.7248194217681885, 1.6449662446975708, 0.7813403010368347, 0.15670013427734375, 1.0839478969573975, 0.553984522819519, -0.5421837568283081, 1.016693353652954, 0.07845883071422577, 0.6362290978431702, 0.9726846814155579, 0.9654582738876343, 1.899163007736206, 0.2182973176240921, -0.14668899774551392, -0.7031407356262207, -0.6027897000312805, 0.6930890679359436, 0.4513877332210541, -0.8346272706985474, -0.5825162529945374, -0.40847155451774597, 1.5894498825073242, -0.8972784876823425, 0.39727333188056946, -1.5898147821426392, -0.19121955335140228, 1.4235073328018188, -1.2102335691452026, 1.1903412342071533, -1.8462027311325073, 0.09528207778930664, -1.0633293390274048, 0.01901480183005333, -0.20140987634658813, 0.8510773777961731, 0.3089255690574646, 0.2063516080379486, -0.5036148428916931, 0.48655936121940613, -0.7586924433708191, -0.6019229292869568, -0.5874203443527222, 0.6054588556289673, 2.2983410358428955, -0.02576146461069584, 1.957404613494873, -0.9767245650291443, -0.5579162836074829, 0.5591204762458801, 0.6481375098228455, 0.801069438457489, 1.616340160369873, -0.16196657717227936, 1.1249645948410034, 0.29025769233703613, -2.780587911605835, -0.9478462338447571, -0.48086994886398315, 0.1566597819328308, 0.5753792524337769, -0.22836971282958984, 1.3561968803405762, -0.9229505658149719, 0.021755414083600044, 0.9650008082389832, -1.075555443763733, 1.3008931875228882, 0.9983919262886047, -0.11492114514112473, 0.05099400505423546, -0.04040123522281647, 0.0683920830488205, 1.291157841682434, 0.5613875985145569, -1.3276869058609009, -1.4929348230361938, 0.6474104523658752, -2.5776150226593018, -1.5289541482925415, -0.570802628993988, 0.016263792291283607, 1.3869352340698242, 0.21359480917453766, 3.5430376529693604, 0.2497892826795578, 0.8905138969421387, 1.0490736961364746, -0.07181970775127411, 0.9846828579902649, 0.6284646391868591, 0.26901647448539734, -0.5988590717315674, -0.39397433400154114, 0.82786625623703, -0.0032894716132432222, 0.4351484179496765, 0.43553125858306885, 0.3101162612438202, 0.047154705971479416, 0.6190009713172913, 0.8614473342895508, 1.4039926528930664, 0.5322123169898987, -0.528978168964386, -0.642220139503479, 0.6691192388534546, -1.4880560636520386, -0.485151469707489, 1.0709654092788696, -0.2962842881679535, -0.618796706199646, 1.2623355388641357, -2.6131718158721924, -0.0430445522069931, -0.42336416244506836, -0.4543963670730591, 0.9381310343742371, 1.5284554958343506, 1.1630781888961792, 0.6534404754638672, 0.35796645283699036, -0.20916731655597687, 0.036517105996608734, -1.5834388732910156, -0.48761776089668274, 0.6277427077293396, -0.08652953058481216, 0.8559591770172119, -1.5165858268737793, -0.5270207524299622, -0.9652652740478516, 1.082616925239563, -0.2285294234752655, -0.2593539357185364, -1.1851441860198975, -0.46822699904441833, -0.9727951288223267, -0.5356854796409607, 2.1979448795318604, -1.1169816255569458, -0.2275369018316269, -0.8591797351837158, 1.3312807083129883, -0.9384475946426392, -0.8350302577018738, 0.39290156960487366, 1.6792041063308716, 0.25836071372032166, -0.541017472743988, -2.5206310749053955, -3.5844645500183105, -0.05720273032784462, 0.048199690878391266, 1.1993061304092407, 0.48744362592697144, -0.2391422688961029, 1.2977532148361206, 1.7428299188613892, -0.6324559450149536, 0.9449775815010071, 1.2103668451309204, -0.7433685064315796, 0.5822667479515076, -0.013637988828122616, 1.1899149417877197, 1.1226704120635986, -2.099175453186035, -0.2080022692680359, 1.387617588043213, 0.5265130996704102, -1.0083043575286865, -0.46258363127708435, -1.0528122186660767, 0.3435070812702179, -1.283640742301941, 1.2366055250167847, 0.8631665706634521, 1.6956835985183716, 1.0935087203979492, 1.0380775928497314, 0.9365491271018982, 0.5971190333366394, -1.9021104574203491, 0.5834382772445679, -0.7524120211601257, -2.3580312728881836, 1.6068954467773438, 0.026157980784773827, -1.5893728733062744, 0.3385200798511505, 0.4669429361820221, 0.22740769386291504, 1.5045219659805298, -1.0760059356689453, 0.9320504665374756, -0.9419798851013184, 0.5046001076698303, 0.6818916201591492, -0.4501282274723053, 2.5535380840301514, -1.7746614217758179, -1.4509872198104858, -0.4346076548099518, -0.8626020550727844, 1.1667430400848389, -0.5529673099517822, 1.3640046119689941, -0.4121207892894745, 0.836799144744873, 0.8300654888153076, 0.14660567045211792, 0.7459577322006226, 0.380903959274292, 2.1447653770446777, 2.410651445388794, -0.30921292304992676, 0.5258227586746216, 1.1899629831314087, 0.6051384806632996, 2.0072734355926514, 2.677976131439209, -1.9439350366592407, -0.511328935623169, 1.3721483945846558, 0.5701846480369568, -0.6720284223556519, -0.19949555397033691, 0.1727953553199768, 1.6986083984375, -0.9121341109275818, 0.2549067735671997, 1.5241605043411255, 1.0937745571136475, -0.0931420624256134, -0.23793412744998932, -1.7494406700134277, -1.6729521751403809, -0.8457815647125244, 0.8755425214767456, 0.8541547060012817, 1.3367605209350586, 0.9500565528869629, 0.7979133129119873, 0.20529170334339142, -0.7635186910629272, 1.2554240226745605, -0.34386709332466125, -1.7838717699050903, 0.4823305308818817, 0.12101549655199051, 2.466702938079834, -0.23714642226696014, 0.38016489148139954, -1.1281780004501343, 0.7105526328086853, 0.28890636563301086, -0.7916283011436462, 0.2642103135585785, -1.2588025331497192, 0.5971057415008545, -0.44453105330467224, 0.374627947807312, -0.8981724977493286, -0.7606667876243591, -0.15165750682353973, 0.8465197682380676, 0.14029012620449066, -2.3733479976654053, 0.6793525218963623, -0.10356657952070236, 0.644068717956543, 1.160842776298523, 0.8357692956924438, -0.7258838415145874, -0.3577231764793396, 0.5077568888664246, 1.0190932750701904, -0.7686432600021362, -0.6319220662117004, -0.9879582524299622, 0.4113899767398834, -0.7186739444732666, 2.3346049785614014, 0.8425583839416504, 1.195311427116394, 0.6160955429077148, -0.5910398364067078, -2.5523478984832764, 1.470738172531128, 0.3168754577636719, -1.8925117254257202, 2.857417345046997, 0.7595958709716797, 1.6143240928649902, -0.23962128162384033, -1.7735244035720825, -0.7770264148712158, 0.8052572011947632, -2.1145687103271484, -0.5269166827201843, 1.1634153127670288, 1.0872116088867188, 0.08793649077415466, -2.569814920425415, -0.25493672490119934, 1.3037420511245728, 0.8854002356529236, -0.3881882131099701, 0.39085161685943604, 0.5884590744972229, -0.38637521862983704, -0.7310280203819275, -0.6946631669998169, -0.8359198570251465, -0.06688866764307022, -0.030821338295936584, 0.5965975522994995, 0.6009596586227417, -0.12646372616291046, -2.136857509613037, 0.5723321437835693, -0.8873356580734253, 1.7576425075531006, -0.2896389663219452, 2.2650320529937744, 0.6562109589576721, 1.6972239017486572, 0.2148139327764511, 0.5325136780738831, -0.407450795173645, -0.4558207094669342, -0.4552024006843567, 0.7678586840629578, -1.1112866401672363, 1.0416642427444458, -0.24414706230163574, 0.6470335125923157, -0.9680948853492737, 0.07737495005130768, -1.6943919658660889, -0.6313762068748474, -0.18627436459064484, -1.0059627294540405, -0.223226860165596, 0.12271004170179367, 0.14007771015167236, -2.0414700508117676, -0.30083709955215454, -0.21195778250694275, -0.8498441576957703, -0.9611077308654785, 1.9240530729293823, 0.8394057750701904, 0.9947872161865234, 1.195285439491272, -0.9367231726646423, -0.25608575344085693, -1.7697302103042603, 0.18869368731975555, -1.9147093296051025, -0.7230842113494873, 0.7446659207344055, 0.5401005744934082, -0.9814414978027344, -0.8986513018608093, 1.3031266927719116, 0.44594690203666687, -0.5042356848716736, 0.1442878097295761, 0.1238861083984375, -1.076439380645752, -0.554142415523529, -2.1052255630493164, 0.23808343708515167, 0.08277343213558197, 1.0373444557189941, 1.0296210050582886, -0.45418041944503784, 1.6009646654129028, -0.7315324544906616, 0.939730167388916, -0.5193395614624023, -0.620844304561615, -1.2433080673217773, -0.06228756159543991, 2.256570339202881, 0.15589439868927002, 0.4730592966079712, -0.3394375145435333, 0.5049340724945068, 0.6251394748687744, 0.4784151613712311, 1.3767495155334473, -0.02247784100472927, 0.42109084129333496, 1.5518392324447632, 0.6295466423034668, -0.744873046875, 1.281235933303833, 0.6465003490447998, -0.7998771071434021, -0.7727622985839844, -0.3857782781124115, 0.8346342444419861, 0.34398818016052246, 0.9958153367042542, 2.4794325828552246, 1.747847318649292, 0.12441655248403549, -0.6500998735427856, 1.407261610031128, -0.694019079208374, -0.7654364705085754, -0.10100942105054855, 0.2969019412994385, -0.24168118834495544, 0.27938079833984375, -2.6746253967285156, -0.3541891574859619, 0.04927453771233559, 0.7242475748062134, 0.3641577363014221, -0.21119356155395508, 2.510408639907837, 0.999980628490448, 0.6375816464424133, 0.33139145374298096, 0.4794115126132965, -0.8001515865325928, -1.0052052736282349, 0.24999484419822693, -1.5738492012023926, 0.6134696006774902, -0.3702804744243622, 0.72633957862854, 0.13713262975215912, -0.8094255924224854, -0.6870294213294983, -0.39463916420936584, 1.3211604356765747, 1.3954188823699951, -0.17094288766384125, 0.15117396414279938, -0.7797881364822388, 1.4296504259109497, 0.2601883113384247, -0.6279731392860413, -1.2920438051223755, 0.5351314544677734, 1.8897227048873901, 0.10749457776546478, -0.8788115382194519, -0.8536117076873779, -0.5556792616844177, -2.2601118087768555, 0.9099363088607788, -0.560704231262207, -1.8212615251541138, -2.9268746376037598, 0.345007985830307, 0.21463781595230103, 1.4253185987472534, 0.9984395503997803, -0.40859562158584595, -1.7188010215759277, 0.20790106058120728, -0.5753607749938965, 0.08453218638896942, -1.5889387130737305, -0.2998954653739929, -0.22755073010921478, -1.2832175493240356, -1.75264310836792, -2.219750165939331, 0.07247970998287201, 1.2008987665176392, -0.040300674736499786, -2.580258369445801, -0.05143554508686066, -1.6800457239151, 0.23618084192276, 1.7267365455627441, 0.34548091888427734, -2.5918996334075928, 1.0434794425964355, 0.32806137204170227, 1.8798422813415527, 0.49978941679000854, 0.6902827620506287, -1.799019694328308, 1.4237289428710938, -1.2426913976669312, -2.018174648284912, 0.5033662915229797, 0.7265804409980774, 1.4438488483428955, 1.1507611274719238, 0.36703798174858093, 0.32908591628074646, 0.8603413105010986, 0.35934245586395264, 0.4574005901813507, -2.0337002277374268, -0.1102357879281044, -0.36842232942581177, 0.5084225535392761, -0.7955746054649353, 0.9359046220779419, 0.7447285652160645, -1.1264729499816895, 0.025520415976643562, 0.5436481237411499, -1.084868311882019, -2.4155595302581787, -0.2500218152999878, 0.2444121539592743, 0.017199071124196053, 1.0841052532196045, 0.29803767800331116, 1.6115491390228271, -0.570375382900238, 1.6535919904708862, -0.2910756766796112, 0.5009312033653259, -1.8803141117095947, -0.7762830257415771, 1.1902472972869873, 0.06678731739521027, 0.010361981578171253, -1.2082505226135254, 0.7952855229377747, -0.9990519881248474, 1.4648138284683228, 0.6153947710990906, -0.5622377991676331, -0.8193256855010986, -0.5262561440467834, 0.37818050384521484, 0.2504798173904419, -1.619558572769165, 0.34786733984947205, -1.0747138261795044, 1.8406562805175781, -1.8914341926574707, 0.9392959475517273, -1.197153925895691, 0.38289394974708557, 1.1850558519363403, 0.4724362790584564, -1.1781024932861328, -1.5447272062301636, 1.4815179109573364, -0.8105404376983643, 0.7845354080200195, 1.2006276845932007, -1.6609454154968262, 1.9807394742965698, -0.1273321956396103, -0.19877490401268005, -0.9882063865661621, 0.23311442136764526, 0.48385265469551086, 1.056944489479065, 1.0433348417282104, 1.0324852466583252, -0.11037301272153854, -0.09188999235630035, 0.12951816618442535, 0.19163034856319427, 1.2744083404541016, 0.9260583519935608, -0.16222140192985535, -0.9224324822425842, -1.916060209274292, -0.21795952320098877, 0.019501537084579468, -1.8780906200408936, -0.3607490062713623, 0.2546367645263672, 0.21745382249355316, -1.4168403148651123, -0.6711909174919128, -1.4620904922485352, 0.8534194231033325, 0.2778284251689911, 2.608417510986328, -0.6026420593261719, 1.5243656635284424, 0.03268630430102348, 0.08564428240060806, 1.0573983192443848, -1.9373326301574707, -1.1975990533828735, 1.5261446237564087, 1.0069515705108643, 1.8757092952728271, 0.5658403635025024, -0.47745105624198914, 0.8596386313438416, -0.4365011155605316, -0.06746035814285278, -0.12113279849290848, -0.8670384883880615, 2.1249022483825684, 0.7252494692802429, 0.13607437908649445, 0.3960166573524475, -0.5698494911193848, -0.3717188537120819, 0.12299338728189468, 0.32667025923728943, -2.1719141006469727, 1.3202437162399292, -2.0562822818756104, -0.9680041074752808, -1.3976435661315918, 0.10635692626237869, -1.7719594240188599, 0.9893237352371216, 0.6910830140113831, 0.9293277263641357, 0.7979755401611328, -0.3330443799495697, 0.24175015091896057, -0.96907639503479, -0.4400922954082489, 0.43155866861343384, -1.2734566926956177, -0.3043397068977356, -0.5072780251502991, 0.888058066368103, 0.5288904905319214, -0.4425778090953827, 0.7964950799942017, 1.9388543367385864, 1.096276879310608, 0.0933796763420105, 0.12725476920604706, -0.973216712474823, -0.5435022711753845, 0.27907440066337585, 0.3771715462207794, -1.4315332174301147, -1.252223014831543, -1.2332903146743774, -1.160212516784668, -0.6441863179206848, -0.16845296323299408, -1.1936999559402466, -0.7124638557434082, -1.994870901107788, -0.2539384961128235, 0.2153652161359787, -0.5748698115348816, -1.5984984636306763, 2.36599063873291, -0.8623560070991516, 0.387052059173584, -2.224165201187134, -0.7924401760101318, 0.09402283281087875, -0.1259334832429886, 0.10445970296859741, 0.0061616250313818455, -0.7984717488288879, -0.9546368718147278, -0.2424338012933731, -0.7110884189605713, -0.658466100692749, 0.3487228751182556, 0.16605930030345917, -0.5205350518226624, 1.0163006782531738, -0.13195392489433289, -0.3093743920326233, 0.760957658290863, 0.181550532579422, -1.6489514112472534, 0.3982948064804077, -1.6703999042510986, -0.3820629119873047, 0.2241460233926773, 0.5866102576255798, -0.9699174761772156, -0.4309283196926117, 0.7094202041625977, 1.9471172094345093, 1.902819275856018, 0.7833452224731445, 1.6065279245376587, -0.5948908925056458, -0.2795731723308563, 0.7550939321517944, 0.7547466158866882, -1.5758415460586548, 1.1248247623443604, -0.9019489884376526, -0.15830595791339874, -1.197464942932129, -0.39931556582450867, 1.016038417816162, -1.3385413885116577, -0.6790691018104553, -0.038533806800842285, 0.5473030209541321, 1.6678781509399414, 0.08769802749156952, 1.0137735605239868, -1.5906102657318115, -0.5659031867980957, -2.4218907356262207, 1.1161198616027832, -0.6145498156547546, 0.5712592601776123, -1.093191146850586, -1.1368608474731445, -1.0485683679580688, -0.749842643737793, -0.34458667039871216, -1.7575623989105225, 0.0414111390709877, 0.5199729800224304, -1.861875295639038, -1.5957642793655396, -1.197749376296997, -0.9671710729598999, 1.6798584461212158, 0.6902992725372314, -1.3196817636489868, 2.663024663925171, 0.7475103735923767, -1.3248032331466675, -1.743302822113037, 0.3325316905975342, -1.7233796119689941, 0.10282749682664871, 0.8172793388366699, -0.9056351780891418, -0.8009445071220398, -0.8977340459823608, 1.248950481414795, 0.5956138968467712, 0.4380834996700287, 0.7920514345169067, -0.07773006707429886, -0.6983513236045837, 0.2639898359775543, 0.8653379082679749, -0.20127259194850922, -0.26872262358665466, -0.9930107593536377, 1.352907419204712, 0.7305850982666016, 0.17557495832443237, 0.16556747257709503, 0.3526182472705841, 0.06217703968286514, 0.9667825102806091, 0.17878633737564087, 0.7529708743095398, 0.05850669741630554, -0.5802467465400696, 1.1388977766036987, 1.1667718887329102, 1.5389353036880493, -0.4277893900871277, -0.6329684257507324, -1.8614089488983154, 0.05647444725036621, -0.8268758654594421, 1.2008172273635864, -1.1632015705108643, 0.9568781852722168, -1.2364312410354614, -0.1177176982164383, 1.2709680795669556, -0.47149476408958435, 0.39011892676353455, 0.7021298408508301, -1.3497400283813477, 2.2599878311157227, -1.3521645069122314, -0.29846620559692383, 0.19918473064899445, 0.41100385785102844, 1.0057315826416016, 1.1937477588653564, -1.3408817052841187, -0.1683889776468277, -2.0791146755218506, 0.2757605016231537, -0.6715865731239319, 0.3488397002220154, -1.7127935886383057, -1.1289076805114746, 0.49591368436813354, 0.7209324836730957, -0.32580047845840454, -0.457854688167572, -0.5625005960464478, -0.5895718932151794, -0.6080636382102966, 2.5658037662506104, 0.7130829691886902, -1.6024072170257568, -0.43358489871025085, -1.024545431137085, 0.15823616087436676, 2.1363377571105957, 1.242508888244629, 0.14431312680244446, 0.4005371332168579, -0.12533962726593018, -1.0109376907348633, -0.597538411617279, 1.1114342212677002, -1.5791317224502563, 1.2760738134384155, 2.3408937454223633, 0.07145849615335464, 1.137880802154541, 0.1933186650276184, -0.27564772963523865, -2.037242889404297, -0.7284643054008484, 1.559209942817688, 0.08894914388656616, -0.310205340385437, 1.5832279920578003, -0.0630025863647461, -1.9606120586395264, -0.5291662812232971, 1.5230520963668823, 1.1610426902770996, -0.316072016954422, 0.8403627872467041, -0.09827636927366257, 2.4841859340667725, 0.7708772420883179, 1.2341911792755127, 0.5275670886039734, 1.1437045335769653, 0.74323970079422, 0.16990184783935547, -0.3748229742050171, -0.26700758934020996, -0.0444413498044014, 0.8545214533805847, 0.7438057065010071, 1.4893542528152466, -0.08204462379217148, 0.28158095479011536, -2.0109128952026367, -0.6208733320236206, -0.5788090229034424, 0.06847189366817474, 0.1984790861606598, -0.4194193482398987, 0.4534952640533447, -0.4088594317436218, 0.0363771952688694, -0.5812825560569763, 0.23909351229667664, -0.551124095916748, -0.18548189103603363, -0.7710667848587036, -1.3114919662475586, 1.3195737600326538, -0.7346956133842468, 1.0865870714187622, -0.32012486457824707, -0.015232699923217297, 0.0892879068851471, -0.8618122339248657, -1.6927355527877808, 0.7885525822639465, 0.22052112221717834, -1.636721134185791, -0.7226472496986389, -0.6365121603012085, 1.0886918306350708, 0.5133900046348572, -0.018810195848345757, 1.1033525466918945, 0.4072924852371216, -0.7357279658317566, -1.3951853513717651, 0.9713966846466064, -1.7120462656021118, 1.1864359378814697, -1.7992291450500488, -0.6686965227127075, 1.211043119430542, 0.8755678534507751, 1.159079909324646, -1.4996263980865479, -0.31722262501716614, -0.07410409301519394, -0.42930421233177185, -0.6066575050354004, 0.27643337845802307, -0.01000590343028307, -0.9770870208740234, -0.5885673761367798, -0.2868419885635376, -0.05276638641953468, -0.28219765424728394, -1.5538517236709595, 0.06785271316766739, 0.4814397394657135, 1.5541962385177612, -0.5245168805122375, 0.8973386883735657, -0.5631275177001953, 1.2553337812423706, -0.13288670778274536, 0.10351976752281189, -0.2072608917951584, 0.1613154113292694, 0.19005785882472992, -1.3765336275100708, 0.6732777953147888, -1.0067921876907349, -0.16962914168834686, 0.9558424949645996, -0.20405058562755585, -0.6415495276451111, 1.6983895301818848, 1.4190880060195923, 1.4384448528289795, 1.0718811750411987, 0.5219932794570923, 0.24017846584320068, -1.4156137704849243, -0.9019532203674316, -0.8231095671653748, 1.1121766567230225, 2.4439971446990967, 0.8278530240058899, -0.170423686504364, 1.4081063270568848, -0.03601275756955147, -0.20406493544578552, -0.04072649031877518, 1.1453052759170532, 1.2806695699691772, 0.32934287190437317, -1.0143349170684814, 0.8835310339927673, -0.7706426978111267, -0.902008056640625, -0.5607842206954956, -1.0693374872207642, -0.7160670757293701, 1.7539781332015991, -0.9805360436439514, -0.07138761878013611, -1.5117987394332886, -0.821042537689209, -0.3673943877220154, 0.10949774831533432, 0.10488089919090271, -0.07412540912628174, -0.7119231224060059, 1.4907256364822388, 0.7163267135620117, -0.5494666695594788, -0.241074338555336, -0.455698698759079, -0.022792333737015724, -0.6422622203826904, 0.7384718060493469, 0.9447696805000305, -0.12790685892105103, 0.36719992756843567, -0.5758433938026428, 1.2699543237686157, -0.3965204954147339, 0.5071042776107788, 0.37386178970336914, -0.9333222508430481, -1.8182196617126465, -0.762016773223877, -1.0573756694793701, -0.6522806882858276, -0.08684450387954712, -0.09038156270980835, -0.5495466589927673, 1.7365121841430664, -1.2332539558410645, 0.9479604959487915, 0.7367309927940369, -0.8318459987640381, 2.0073864459991455, 1.3258764743804932, 0.33327996730804443, 0.4792458713054657, 0.07478315383195877, 0.6380620002746582, -0.4972573518753052, 1.1710660457611084, 1.0253074169158936, -0.3666056990623474, -1.485581874847412, 1.1606673002243042, 0.20228974521160126, -0.8818290829658508, 1.4737221002578735, 0.22606192529201508, -1.5169410705566406, 0.9616043567657471, -1.1634855270385742, -1.9017298221588135, -0.7434409260749817, -0.21291783452033997, -1.8864433765411377, 2.3671247959136963, 0.28997206687927246, 0.20578743517398834, -0.20330671966075897, 0.29932501912117004, -0.8330298066139221, -0.28392890095710754, 1.1698709726333618, -0.2862740159034729, 0.22469429671764374, -0.2165294587612152, 1.9224507808685303, 1.23215913772583, 0.4450613856315613, 0.7926515936851501, 0.37791576981544495, 0.33248457312583923, 2.796947479248047, -2.2408392429351807, -0.14594390988349915, 0.06401021778583527, 0.9883536696434021, -1.517061710357666, -0.6423249244689941, 0.4277935326099396, -1.9352651834487915, 0.5749775767326355, 2.235380172729492, -0.9469984173774719, -2.065579652786255, 0.7013248801231384, -0.27445995807647705, 2.143908977508545, -0.39855459332466125, -0.6417160034179688, 1.2213025093078613, 1.331790566444397, 1.1363247632980347, 2.3492887020111084, 0.625083863735199, 0.07349542528390884, -0.5230858325958252, -0.10912200808525085, -0.45288896560668945, -0.5412031412124634, -1.0778834819793701, -2.388991594314575, 0.4621421694755554, 0.4371509850025177, -1.7099487781524658, 0.9079504609107971, -2.8377530574798584, -0.5012834668159485, 0.1580505520105362, -0.35745254158973694, 0.7663711905479431, 1.2538657188415527, 1.242722988128662, 0.13456971943378448, 1.6645251512527466, -0.3650932312011719, -0.5505802035331726, -0.6419580578804016, -0.7297185063362122, -0.41791266202926636, -1.3465346097946167, -1.6593711376190186, 1.5958982706069946, 0.6155574917793274, 1.0175031423568726, 0.3708692193031311, -1.0681431293487549, 1.3424733877182007, 0.7730174660682678, -1.227430820465088, 0.7871042490005493, -2.1277594566345215, 1.047837495803833, -0.8636338710784912, 1.0577000379562378, -0.8564956188201904, 1.1116591691970825, 0.38216203451156616, 0.6298742890357971, 0.30002835392951965, 0.5550928115844727, 0.7000190019607544, 0.8188809156417847, -2.3078105449676514, -0.3860924243927002, -0.5445693135261536, -0.3070026934146881, -0.14081896841526031, 2.068286418914795, 0.5758486390113831, -1.4435968399047852, -0.4586981534957886, 1.001440167427063, -1.1170812845230103, 0.23886314034461975, -0.5986393094062805, 0.37026718258857727, -0.2082880139350891, -0.03453323245048523, -0.2704053521156311, -0.8501878976821899, 1.2763862609863281, 0.39090201258659363, 0.127121239900589, 1.5881447792053223, 0.28787338733673096, -0.030354520305991173, 0.22298510372638702, -0.39855870604515076, -0.8161457777023315, 1.89461088180542, -0.8175868988037109, 1.278489351272583, 0.4841803312301636, -1.173993706703186, -0.2429685890674591, -0.7764348983764648, -0.06656701862812042, 0.8569794297218323, -0.9269225597381592, -0.9074987769126892, -1.2219282388687134, 1.498293161392212, 0.12948356568813324, 0.34813180565834045, 0.1666726917028427, -1.169100046157837, 1.6975351572036743, -1.141953945159912, -2.238913059234619, 0.7431966066360474, -0.929270327091217, 1.0217982530593872, 0.714447021484375, 0.11888036131858826, 0.6983495354652405, 0.770033061504364, -0.5231799483299255, -0.7880510091781616, -1.329290509223938, -0.43187710642814636, -0.7718830704689026, -1.1236203908920288, -0.7599384188652039, 2.5893547534942627, 0.651188313961029, -1.4156911373138428, 1.6827160120010376, -1.8601897954940796, -0.136321023106575, -1.1217654943466187, -1.108107089996338, -0.5309687852859497, 0.4733832776546478, 1.1485931873321533, -1.897813320159912, -0.4746876358985901, -0.895353376865387, 0.5254103541374207, 0.8141991496086121, -1.3237956762313843, -0.8701514005661011, -0.8324870467185974, 0.7285921573638916, -0.49559661746025085, -0.8622303605079651, -0.7319139242172241, -0.7962369322776794, 0.07172638922929764, 1.7861484289169312, -1.816834568977356, -0.7733543515205383, 0.3606375455856323, -2.3214900493621826, 0.3697643280029297, 0.3699345886707306, -0.0044198487885296345, -1.419907569885254, 1.797398328781128, 0.16916128993034363, -0.2941310703754425, 1.1454647779464722, -1.6133061647415161, -0.8350261449813843, -1.974084734916687, -0.23795346915721893, 0.07282033562660217, -0.37092065811157227, -0.2899620532989502, 0.6528443694114685, -0.73223477602005, -0.35075896978378296, -1.3670276403427124, 0.5532200932502747, -0.3557664155960083, 1.6331441402435303, -0.6340839266777039, 0.5070893168449402, 0.3408569395542145, -1.3321740627288818, 0.7329670786857605, 0.4675173759460449, 0.9785704612731934, 0.8971565365791321, -0.31600746512413025, 1.5839923620224], "dtype": "float32"}], "e": 3.5430376529693604}, {"a": [{"__nd__": [40.332767486572266, 78.8197021484375, -208.38771057128906, -200.58677673339844, -181.60292053222656, -11.146848678588867, -2.09833025932312, 59.27072525024414, 6.871888637542725, -171.2584991455078, -48.345314025878906, 146.5140380859375, 109.44557189941406, -160.30099487304688, -32.79180145263672, -148.45526123046875, 39.01753234863281, 102.8442153930664, 81.97654724121094, -110.34584045410156, -53.022804260253906, -16.193527221679688, 175.5861053466797, 30.10441017150879, 11.18846321105957, 13.028736114501953, 21.534181594848633, -85.66217041015625, -114.31985473632812, -55.24164581298828, -30.63527488708496, -65.23127746582031, 70.24655151367188, 1.7215380668640137, 76.82072448730469, -73.87203979492188, -250.23992919921875, -53.350425720214844, 88.56715393066406, 18.127025604248047, -259.322509765625, -58.30081558227539, -74.56207275390625, -31.037389755249023, -72.37747955322266, -9.977691650390625, -63.6982421875, -51.75726318359375, 92.11836242675781, 23.7974910736084, -19.754104614257812, -143.4989013671875, -89.66085052490234, -63.69910430908203, 40.17646408081055, -167.7703094482422, 22.31930923461914, 137.16220092773438, 20.822406768798828, -141.36355590820312, 99.13966369628906, 1.7262390851974487, 63.99972152709961, 95.02632141113281, 66.27886962890625, -14.18604564666748, -89.31775665283203, 224.1505584716797, -82.6938705444336, 90.45872497558594, -90.6201400756836, 130.88943481445312, 195.91957092285156, 91.25440216064453, -71.50051879882812, -31.525718688964844, 152.43003845214844, 153.98416137695312, 68.37772369384766, 27.478544235229492, 163.10177612304688, 59.03589630126953, 38.86369323730469, 93.5613021850586, 42.68684005737305, 124.0982666015625, 155.97608947753906, 55.42755126953125, -231.82843017578125, 6.133068561553955, -42.36565017700195, -110.7127914428711, -44.537322998046875, 136.59889221191406, -139.75360107421875, -33.0943489074707, -69.56726837158203, -148.5043487548828, -148.13821411132812, 56.44017791748047, -131.05068969726562, 187.3779754638672, -17.64215850830078, 36.965553283691406, 57.92036437988281, -156.07070922851562, 108.51687622070312, 18.349777221679688, -5.573485374450684, 58.150108337402344, 19.042985916137695, 6.299126625061035, -122.6818618774414, -9.67529296875, 65.13888549804688, -79.76000213623047, 81.78561401367188, -32.97694778442383, 56.54785919189453, 6.295629501342773, -101.39226531982422, -19.818050384521484, -55.467403411865234, -80.1070327758789, -83.06614685058594, -213.75918579101562, -31.61543083190918, -15.853561401367188, -106.43061065673828, -161.13668823242188, 63.88094711303711, 34.49711990356445, 110.53842163085938, 106.0733642578125, -98.72229766845703, 0.60111004114151, -14.337396621704102, 87.88108825683594, -96.38579559326172, -92.12532043457031, 125.14956665039062, -109.84590148925781, 10.447307586669922, 17.372961044311523, -148.37582397460938, -70.96300506591797, -86.22817993164062, -53.76877212524414, -107.77265930175781, 91.72459411621094, -32.212249755859375, 93.26864624023438, 84.3515625, -174.12167358398438, 84.1761245727539, -54.355987548828125, 71.08392333984375, 98.10962677001953, 130.0045166015625, 99.13491821289062, -12.238753318786621, -59.0514030456543, 11.537322044372559, 24.891464233398438, 5.770569324493408, -20.709142684936523, 58.9561767578125, -38.7393913269043, 27.945199966430664, 16.62137222290039, 112.89622497558594, -58.0706787109375, -59.03431701660156, -85.22908782958984, 79.05972290039062, 39.94415283203125, -25.21337890625, 173.1917266845703, -89.2450180053711, 84.30670166015625, 40.805091857910156, -38.31961441040039, -104.61495971679688, -74.81102752685547, 85.60652160644531, -143.31549072265625, -27.392515182495117, -83.72604370117188, 2.1118052005767822, -75.79536437988281, -48.8140983581543, -136.31590270996094, 15.051445960998535, -86.39385986328125, 1.3623547554016113, -179.1341552734375, -127.90045928955078, -41.30018615722656, -79.44734954833984, -25.389646530151367, -66.29338836669922, 67.42764282226562, 46.66413879394531, 104.39781951904297, 61.790199279785156, 112.84162902832031, 111.88491821289062, 31.379501342773438, -46.09452438354492, -7.417035102844238, 13.502639770507812, -76.11386108398438, -42.13727951049805, 101.40686798095703, 71.74562072753906, -76.76675415039062, -98.18804931640625, -55.21361541748047, -165.05909729003906, 99.40154266357422, 40.671112060546875, -118.26721954345703, -28.347318649291992, -70.48225402832031, 125.88847351074219, -127.19426727294922, -70.16236114501953, 159.36875915527344, 165.16796875, 158.19830322265625, 33.99025344848633, -45.5079460144043, -84.17765808105469, 76.98847961425781, -78.95463562011719, -11.766523361206055, -15.93504810333252, 99.9498291015625, -68.6773681640625, 98.62896728515625, -57.189979553222656, -49.78432083129883, -110.66346740722656, 24.022512435913086, -16.395355224609375, 9.59548282623291, -32.28030776977539, 9.82197380065918, 164.90960693359375, 46.774173736572266, 101.11498260498047, -3.908280849456787, -1.9753397703170776, 118.05463409423828, 270.6997985839844, -56.14765167236328, 49.506324768066406, -45.81010818481445, 53.977718353271484, 21.497447967529297, -53.595314025878906, -151.4670867919922, -159.47984313964844, 133.2866668701172, 8.029218673706055, 3.4501218795776367, 60.440616607666016, -4.382202625274658, 213.27151489257812, 16.250829696655273, 104.0649185180664, 22.493453979492188, -142.69749450683594, 58.14372634887695, 30.717012405395508, -42.89567565917969, 140.03562927246094, 68.68408966064453, 84.29736328125, -118.56246185302734, -43.67891311645508, -164.8009033203125, 62.639102935791016, 109.61965942382812, -56.2407341003418, 234.75914001464844, -105.61238098144531, 27.56458282470703, -208.8780059814453, -198.14151000976562, 38.466957092285156, 14.990582466125488, 204.9001007080078, 26.843612670898438, 135.7608642578125, -73.39189147949219, 65.31333923339844, 144.12437438964844, 12.19018840789795, 143.99827575683594, 113.41078186035156, 31.716121673583984, -99.42716217041016, -57.158504486083984, -162.39468383789062, 102.30384063720703, 155.97801208496094, 34.5087890625, 46.066383361816406, -193.86209106445312, 114.80703735351562, -132.5418243408203, -52.25912857055664, 203.83995056152344, 52.31010437011719, -71.01239776611328, -80.89675903320312, 31.88669776916504, 118.07032012939453, 26.48189353942871, 15.74429988861084, 53.614051818847656, 75.15402221679688, -150.1982421875, -178.81344604492188, 133.69236755371094, -69.89886474609375, -211.6579132080078, -47.6353645324707, -15.916401863098145, -32.15812683105469, 121.22578430175781, -109.72261810302734, -4.261977195739746, -143.19314575195312, 208.78134155273438, -117.2350845336914, -164.59803771972656, 71.18059539794922, 22.03470802307129, -126.29014587402344, 57.68149185180664, 77.11194610595703, -43.52603530883789, 55.31687927246094, 93.42617797851562, 68.3935546875, -47.56865310668945, -6.457054138183594, -45.92633056640625, -41.730464935302734, -6.4722065925598145, -46.38547897338867, -17.0682373046875, -14.564346313476562, -22.978986740112305, 53.1694450378418, -118.06266021728516, -87.19335174560547, -60.283302307128906, -85.58589172363281, 47.81332778930664, 23.166213989257812, -6.634406566619873, -27.579572677612305, 52.84015655517578, 82.7677001953125, 88.60005950927734, -22.702281951904297, 5.842177867889404, -168.23109436035156, -24.150312423706055, 212.77525329589844, -72.19654083251953, 93.15422058105469, 61.67036056518555, -96.63040924072266, -181.66522216796875, -80.82622528076172, 127.29273986816406, 103.35558319091797, 62.944881439208984, -41.64833450317383, 67.19072723388672, -113.88333129882812, 86.626708984375, 42.53689956665039, 70.13697814941406, 304.1808776855469, 279.65203857421875, -84.05587005615234, 24.16116714477539, -1.4524023532867432, 33.68405532836914, 237.2355499267578, -111.7590103149414, -7.31359338760376, 135.81373596191406, -56.69440841674805, 164.69625854492188, 12.543560981750488, -6.365928649902344, 64.44168090820312, -21.281465530395508, 102.31336212158203, 177.23080444335938, -242.5402374267578, 107.37938690185547, -191.5401611328125, -181.9390106201172, 84.2120132446289, 39.24622344970703, 39.966636657714844, -86.67198181152344, -55.41118240356445, -28.670902252197266, -163.17820739746094, 50.477622985839844, -24.204898834228516, 56.90424728393555, -244.2229461669922, -257.52313232421875, -60.26793670654297, -24.536706924438477, 50.689632415771484, -52.487281799316406, -82.21061706542969, -98.00125122070312, 91.04640197753906, 10.770312309265137, 171.45326232910156, -8.820621490478516, 38.82667541503906, -11.4226655960083, -31.35905647277832, 33.816162109375, -119.30473327636719, -50.50275421142578, 38.806549072265625, -76.48291015625, -215.75877380371094, -57.433162689208984, -116.41981506347656, 39.249855041503906, -129.26898193359375, -94.22195434570312, 5.112799167633057, -47.24334716796875, 53.12844467163086, 10.164063453674316, -23.883602142333984, 58.98530960083008, 27.663856506347656, -59.96166229248047, 93.4705810546875, -5.747570514678955, 41.67570114135742, -9.198575973510742, 37.61217498779297, -62.187801361083984, 40.6775016784668, 16.356945037841797, -16.659263610839844, 3.2122457027435303, -31.984783172607422, 52.1646728515625, 154.015625, -79.0248794555664, -23.36841583251953, -145.43162536621094, -33.06570053100586, -1.8986856937408447, 38.331146240234375, -125.80193328857422, -71.16372680664062, 0.9717715978622437, 289.4532775878906, 14.78067398071289, 11.597201347351074, -72.00833129882812, -39.169776916503906, -243.76914978027344, -72.5876235961914, -79.16590881347656, -37.988250732421875, -45.74677276611328, 205.78135681152344, 94.82308959960938, -12.670289039611816, -192.5359649658203, 171.39100646972656, -59.28947448730469, -89.67542266845703, -16.522586822509766, -17.783090591430664, 137.80368041992188, 6.516106605529785, 44.49095916748047, 58.67453384399414, 148.81126403808594, 66.15608978271484, -15.77009391784668, -42.2122802734375, -141.4153289794922, -100.10494232177734, -304.5227355957031, 103.307861328125, -77.34595489501953, -58.307125091552734, 69.91354370117188, -36.91132736206055, 23.85948944091797, -67.49811553955078, -32.45817947387695, 0.43094539642333984, 158.99134826660156, -163.161376953125, -88.843017578125, 33.17424774169922, 57.79879379272461, -35.144466400146484, -5.376312255859375, -11.737789154052734, 68.0020980834961, -148.87759399414062, -15.335381507873535, 132.54249572753906, 99.88937377929688, -65.3120346069336, 41.468994140625, -109.36578369140625, 48.76522445678711, 39.43037033081055, 0.4936355948448181, 33.71987533569336, -23.300107955932617, 109.75115966796875, 87.91930389404297, 66.43382263183594, 111.92815399169922, 115.77076721191406, 13.879485130310059, 62.881107330322266, -4.798396110534668, -169.68453979492188, 119.82630920410156, -14.099064826965332, -8.27623176574707, 201.6609344482422, -14.843786239624023, -32.232948303222656, -179.1118927001953, 33.08659362792969, 13.306527137756348, -63.17475891113281, 108.86126708984375, -43.68003845214844, -129.2814178466797, -65.56351470947266, -25.694257736206055, -21.911968231201172, -0.7533632516860962, -114.52378845214844, -96.18180847167969, -189.02125549316406, -18.154420852661133, 146.58131408691406, 97.82484436035156, -147.4577178955078, -61.75505065917969, 120.35137939453125, -44.867042541503906, -47.79802703857422, -140.4379425048828, 175.43905639648438, 101.83106231689453, -108.67610931396484, 92.33622741699219, -97.1572036743164, 8.269615173339844, 119.27883911132812, -92.37930297851562, -92.63288879394531, -107.1787109375, -9.740139961242676, 79.75312805175781, 80.37743377685547, 161.0862274169922, 181.54600524902344, 62.8101692199707, -155.1089324951172, -103.33486938476562, 49.00722122192383, 18.129579544067383, -20.465539932250977, -60.892547607421875, 37.5959587097168, -151.40380859375, -27.821290969848633, 34.151153564453125, 125.85682678222656, 125.9896011352539, -72.83245849609375, -129.07635498046875, -4.927917957305908, 31.073884963989258, -155.0637969970703, -159.7062225341797, -76.13484191894531, -20.387405395507812, 22.516328811645508, -23.103412628173828, -11.593269348144531, 86.85443115234375, -37.44997024536133, -41.39152145385742, -160.36187744140625, 25.40867042541504, 95.45295715332031, 58.56259536743164, -89.6536636352539, 106.11566162109375, -33.628013610839844, 172.51681518554688, 22.182186126708984, 116.36060333251953, 19.327409744262695, 10.709465026855469, -80.70469665527344, -76.35797882080078, 108.15951538085938, -98.94365692138672, 156.96873474121094, -179.92752075195312, -8.37343692779541, -56.246063232421875, 169.10052490234375, -52.43498229980469, -48.91621398925781, -86.12989807128906, 63.802825927734375, -23.994136810302734, -8.621274948120117, -158.62594604492188, -37.554656982421875, -35.36260986328125, -71.31007385253906, 168.8046112060547, 237.80172729492188, -12.507225036621094, -53.59636688232422, 65.69599914550781, 75.47865295410156, -95.25997924804688, 55.621768951416016, -57.45074462890625, -44.87312316894531, -19.477737426757812, 86.56375122070312, 11.707557678222656, 35.04158401489258, 14.511881828308105, -181.5699920654297, 132.16587829589844, -9.933432579040527, -55.013980865478516, -99.05555725097656, -140.285400390625, 20.9344425201416, 105.11483764648438, -8.692547798156738, -36.12525177001953, 34.82302474975586, -322.44915771484375, 12.477957725524902, -43.85205841064453, 66.1723403930664, 111.08675384521484, -29.01525115966797, -120.61835479736328, 69.53929901123047, 2.6464622020721436, 60.57181167602539, 78.59053039550781, -1.5358809232711792, -69.6843490600586, 30.1245059967041, 9.845366477966309, 139.40821838378906, 15.971992492675781, 122.7770767211914, 92.98015594482422, 21.374496459960938, -229.8224334716797, -111.27679443359375, 73.42526245117188, 90.73333740234375, -72.76571655273438, -163.37405395507812, 238.43702697753906, -16.07756996154785, -192.3143768310547, 51.00880813598633, 153.6552734375, -217.6708984375, 7.462306022644043, -49.38481140136719, -1.8738248348236084, 65.13833618164062, -98.98629760742188, -112.9815444946289, -125.02506256103516, 99.31124877929688, 112.7929458618164, -112.67832946777344, 116.93927001953125, -65.16362762451172, 5.17421817779541, -46.72077941894531, -27.50493049621582, 145.02935791015625, 19.528568267822266, -82.19290161132812, 166.8402862548828, 195.58251953125, 12.453432083129883, 167.90505981445312, 187.4681396484375, -62.372337341308594, -166.09353637695312, -127.99202728271484, -143.85845947265625, -48.778499603271484, 190.1310272216797, -43.37600326538086, -55.337928771972656, 156.4282684326172, -22.48379898071289, 49.692996978759766, 214.49473571777344, -146.9001922607422, -115.78397369384766, -27.947153091430664, 20.455739974975586, 10.837044715881348, 52.60710144042969, -119.09754180908203, -40.852386474609375, -124.74752807617188, -13.353645324707031, 51.895591735839844, 58.97164535522461, 23.320581436157227, 110.37718200683594, -142.86058044433594, -6.824306011199951, 37.611385345458984, -155.4439697265625, 282.4267578125, -64.12263488769531, 100.05718231201172, -30.228050231933594, 2.481426954269409, -3.55647349357605, 171.9240264892578, -2.569401741027832, 73.71478271484375, 76.62299346923828, 52.624324798583984, 78.63945007324219, -79.85987854003906, -177.78001403808594, -34.17258071899414, -83.53809356689453, -218.9209747314453, 127.46625518798828, -184.89303588867188, 175.20184326171875, -144.49436950683594, -82.26516723632812, 75.35606384277344, 61.862552642822266, 119.30792236328125, 12.5933837890625, -25.675195693969727, -262.613525390625, 40.20500183105469, -21.299531936645508, -8.540154457092285, 51.78050231933594, 146.97988891601562, -84.04071044921875, -21.909770965576172, -33.166603088378906, -183.87989807128906, 101.5121841430664, -35.95835494995117, -187.6530303955078, 23.853052139282227, -5.28184175491333, 17.381885528564453, -143.93699645996094, -131.85142517089844, -2.1586737632751465, 75.04955291748047, -106.04659271240234, 0.30895674228668213, -36.22784423828125, 22.790958404541016, 79.29930114746094, 63.999046325683594, 123.00634002685547, 186.59861755371094, -60.619903564453125, 123.26486206054688, -35.939266204833984, -17.261764526367188, -56.0377197265625, 53.44216537475586, -159.8730010986328, -130.88809204101562, -80.61280059814453, -120.61843872070312, -189.28346252441406, 5.166651725769043, -39.73847579956055, -100.32392120361328, 44.49435806274414, 150.66033935546875, -116.08433532714844, 102.53620910644531, -90.48136138916016, 107.7436752319336, 86.06974029541016, -108.83678436279297, 197.5586395263672, -96.30612182617188, -161.13311767578125, -74.34550476074219, 25.38823699951172, -71.23774719238281, -112.74149322509766, 95.29920959472656, 54.98971176147461, -49.3900032043457, 196.70010375976562, -166.90130615234375, -70.00252532958984, -112.76411437988281, -43.556766510009766, -0.08397594839334488, 70.66558074951172, -59.110076904296875, 43.40110778808594, -45.54445266723633, 173.0695343017578, -141.55487060546875, -33.95735168457031, 61.1468391418457, 105.99368286132812, 42.11975860595703, 28.661819458007812, 175.5216522216797, 4.6052751541137695, 15.270028114318848, -70.63516998291016, 101.5310287475586, 7.744636058807373, -169.92025756835938, -164.4834747314453, -231.76034545898438, 23.083839416503906, 30.43059539794922, 113.81005859375, -67.72985076904297, -268.0806579589844, 14.815361976623535, -27.729276657104492, -37.07493591308594, 53.46999740600586, -128.4813690185547, -12.828215599060059, -25.826635360717773, 35.47111129760742, 149.35641479492188, -115.2554702758789, 36.935264587402344, 4.079541206359863, 15.09319019317627, 145.4278106689453, 156.26797485351562, -64.0462875366211, 54.60312271118164, -2.1851837635040283, -88.02084350585938, 69.93321990966797, 37.31306457519531, 110.35387420654297, 25.564964294433594, -40.814308166503906, -42.048362731933594, -6.222017765045166, 77.5909423828125, -40.318241119384766, 21.780799865722656, 58.26944351196289, 224.2373809814453, 46.361942291259766, 107.5279541015625, -15.474205017089844, -43.175750732421875, 87.98948669433594, -81.45606994628906, -73.30245208740234, 103.69955444335938, -26.421981811523438, 21.68575096130371, -99.72425079345703, -31.328731536865234, -58.6793327331543, -75.65608978271484, 53.01103210449219, -56.4583625793457, -86.80528259277344, -59.03776168823242, -33.9178581237793, 156.40390014648438, -83.52388763427734, -19.577409744262695, 77.97224426269531, -150.19390869140625, -11.768701553344727, 23.592321395874023, 65.03839874267578, -7.18513298034668, -24.243684768676758, 0.7565304636955261, -75.60111999511719, -95.2699966430664, 136.89967346191406, -31.480987548828125, -111.44197845458984, 75.61405181884766, -125.72342681884766, -106.52713775634766, -77.0067367553711, 146.02688598632812, 10.483369827270508, 125.96626281738281, 9.520581245422363, -4.037799835205078, -62.0695686340332, 2.708575487136841, 93.03047943115234, 62.08169937133789, -51.761878967285156, -43.88252258300781, 64.94749450683594, -41.32282257080078, -23.011192321777344, -144.0519256591797, -77.10946655273438, 60.93821334838867, 58.66888427734375, 23.318172454833984, 82.93366241455078, -128.87176513671875, -20.597200393676758, -56.6463737487793, -9.592663764953613, 230.61990356445312, 64.44377899169922, -169.30496215820312, 8.719131469726562, 124.33866882324219, 195.63160705566406, 76.08733367919922, 61.580718994140625, -68.47764587402344, -94.63899230957031, -0.4995332956314087, 19.82221794128418, 163.34719848632812, 190.1059112548828, 107.3566665649414, -55.01655197143555, -48.032840728759766, -65.36503601074219, 29.37511444091797, -70.32968139648438, -151.03538513183594, -92.32295989990234, -33.6478271484375, 93.20165252685547, 87.84965515136719, -19.79023551940918, -126.7428207397461, -141.23370361328125, 126.7585220336914, -12.05179214477539, 56.84721374511719, -10.699751853942871, -33.48550033569336, 66.900634765625, -17.155961990356445, 53.674774169921875, -194.09835815429688, 39.321266174316406, 281.9967956542969, -3.92972993850708, -11.485201835632324, -113.53592681884766, 41.42841720581055, -111.34716796875, -77.22303771972656, -61.98147964477539, -148.298095703125, 35.70759963989258, -188.58694458007812, 70.3945541381836, -150.3037872314453, -82.65016174316406, 12.588115692138672, -87.25082397460938, 192.02243041992188, 8.086372375488281, 70.53170013427734, 21.664794921875, 99.09522247314453, -50.6332893371582, 42.55485534667969, 54.41550827026367, 152.32249450683594, 82.54519653320312, -135.80514526367188, -108.64801788330078, 123.22602844238281, 152.7033233642578, 129.61428833007812, 179.53956604003906, 105.0501480102539, 127.56430053710938, 96.03399658203125, -18.6169376373291, -115.25947570800781, 31.16554069519043, -8.653487205505371, -66.50367736816406, -52.17848205566406, -32.471012115478516, 6.3666229248046875, -5.450493812561035, 59.66482925415039, -34.03601837158203, 229.5385284423828, 137.4788818359375, -20.455045700073242, 42.720252990722656, -48.682010650634766, 50.6194953918457, 293.2431335449219, 4.721845626831055, -55.219905853271484, 31.087263107299805, 70.38330841064453, 137.6873779296875, -75.87260437011719, -66.22610473632812, 42.15858840942383, -65.19075775146484, 340.5093078613281, -247.11322021484375, -32.33216857910156, -42.71334457397461, -86.74018859863281, 67.76543426513672, -15.263226509094238, -7.003601551055908, 130.31231689453125, 47.329288482666016, 5.4157395362854, -70.80596923828125, 122.11116790771484, -99.52792358398438, 129.598388671875, -56.7750129699707, 244.3694610595703, 31.86004066467285, 21.15015983581543, -8.053342819213867, 6.522689342498779, 96.24903869628906, -50.196475982666016, -52.60521697998047, -114.39730072021484, 63.474517822265625, -97.97247314453125, -93.0744400024414, 153.39456176757812, -210.33335876464844, -209.5374298095703, -30.90737533569336, 9.49022388458252, 111.45062255859375, 40.33742904663086, 2.4968111515045166, 91.1670913696289, -147.80459594726562, -123.86686706542969, 3.2764065265655518, 239.97853088378906, -40.90713882446289, -10.317167282104492, -50.80703353881836, -106.29882049560547, -182.43927001953125, 59.121559143066406, 27.421361923217773, -52.501808166503906, 2.8891823291778564, 51.28256607055664, -64.38461303710938, -82.33341217041016, -162.43028259277344, -61.45217514038086, 114.93997192382812, -93.80699920654297, -37.3010368347168, 55.96039962768555, 26.26334571838379, 48.34813690185547, 61.448486328125, 75.45741271972656, 4.408045291900635, 13.646672248840332, 179.81910705566406, -4.836972713470459, 65.5145263671875, -77.04393005371094, 88.04508209228516, 112.79473876953125, 20.963092803955078, 37.32384490966797, -58.64094161987305, -56.379844665527344, 23.72441864013672, 161.8418426513672, -55.97364044189453, 184.8016357421875, 83.46541595458984, 11.916391372680664, -79.9556655883789, 172.498291015625, 47.664241790771484, 56.44365692138672, 143.55108642578125, -7.599560737609863, -134.41278076171875, 119.18145751953125, -162.81576538085938, -112.18360137939453, -39.11405563354492, 79.16488647460938, -39.816043853759766, 160.0935821533203, -103.9705810546875, 135.5596160888672, 96.79147338867188, -41.085697174072266, -90.31884002685547, -56.88915252685547, -92.0672607421875, 154.31288146972656, -39.017173767089844, -84.50312042236328, 51.8825569152832, -81.42350769042969, -185.12635803222656, 13.702903747558594, -60.243099212646484, 118.91900634765625, 173.77464294433594, 35.04395294189453, -24.2861385345459, 95.00398254394531, 20.662633895874023, -173.3855743408203, -29.30853843688965, -97.47093963623047, -8.813578605651855, -175.52127075195312, 202.44590759277344, -25.080886840820312, 11.083015441894531, -57.67167663574219, 105.896728515625, -27.5465087890625, -90.94944763183594, -184.48675537109375, 132.016357421875, -119.41625213623047, 136.67373657226562, 189.2049102783203, -40.41399383544922, -91.60735321044922, 159.9248809814453, -70.1930160522461, 39.86811447143555, -289.88427734375, -131.4856414794922, 45.66376495361328, 129.64602661132812, -61.44024658203125, 126.92778778076172, 131.59910583496094, 25.6291446685791, 120.057861328125, -106.99063873291016, 32.84361267089844, 187.8542022705078, -158.05760192871094, 66.45386505126953, 37.33700942993164, -76.61038970947266, -153.0394287109375, -144.40457153320312, -116.12527465820312, -70.63549041748047, -29.055965423583984, 22.663166046142578, 113.93317413330078, 11.883158683776855, -31.644596099853516, -182.5946044921875, -22.50324058532715, -163.3430938720703, -3.1294307708740234, -24.19389533996582, 65.34927368164062, -91.80906677246094, -71.61225891113281, -75.32540893554688, -28.477304458618164, -83.80352783203125, 130.07833862304688, -75.55516052246094, 144.3355255126953, -136.3839874267578, 20.391498565673828, -162.6246337890625, 79.18094635009766, 27.194732666015625, 87.10519409179688, 181.58839416503906, 137.83802795410156, 49.08565902709961, 65.60581970214844, -12.995612144470215, 91.12417602539062, -164.61380004882812, 77.47710418701172, -166.66624450683594, -53.074710845947266, 110.76667022705078, -35.727561950683594, -22.93250846862793, -45.1607551574707, 41.63307189941406, -141.23289489746094, 17.23415184020996, 155.04701232910156, -33.714473724365234, 23.556062698364258, 35.60758590698242, 29.527137756347656, -15.200407028198242, -139.822998046875, -39.6374397277832, -115.10423278808594, 82.91864776611328, 6.930073261260986, -106.249755859375, 167.738525390625, -150.71109008789062, -61.699363708496094, 213.53631591796875, -142.77308654785156, -170.385009765625, -29.270198822021484, -45.12297821044922, -169.39817810058594, 190.3306427001953, 159.9114990234375, 18.93842315673828, -162.91064453125, -89.88388061523438, 276.2381896972656, 72.10247802734375, 50.76454162597656, -110.57272338867188, 77.42037963867188, 120.21660614013672, -33.413177490234375, -91.18501281738281, 47.65254592895508, -161.8527374267578, -74.08373260498047, 170.61264038085938, 95.21316528320312, 53.484275817871094, 83.03726959228516, -118.11598205566406, 105.64639282226562, 193.61602783203125, 225.37229919433594, 10.663116455078125, 27.886783599853516, -56.01019287109375, 101.93589782714844, 122.2535629272461, 97.69283294677734, -30.869789123535156, -53.86741256713867, 223.95814514160156, 59.847267150878906, 126.2251968383789, -28.401145935058594, -135.5360870361328, 57.73509216308594, 87.80901336669922, -148.33187866210938, -117.880126953125, -15.793804168701172, 72.47248077392578, -18.487268447875977, -32.2767219543457, -151.16770935058594, -118.50020599365234, -2.1176013946533203, -5.999139785766602, -40.69407272338867, -50.01567077636719, -8.301470756530762, -149.0072784423828, 133.2055206298828, 136.67771911621094, -66.99172973632812, -90.8704605102539, -63.93089294433594, 18.899152755737305, -123.25163269042969, 8.752259254455566, 9.211654663085938, 39.27516174316406, 95.63742065429688, -13.867644309997559, -95.37501525878906, 195.88186645507812, 86.80162048339844, -15.328404426574707, 24.69202995300293, -184.60472106933594, -103.62040710449219, -37.061519622802734, -6.620851993560791, 2.222320079803467, 33.45968246459961, -53.68839645385742, -281.1500549316406, -23.335657119750977, 26.786823272705078, -105.26588439941406, -24.56428337097168, 107.63542938232422, -4.225011348724365, 22.5114688873291, -234.0111541748047, 58.6502571105957, -127.94163513183594, 126.5932846069336, -172.86871337890625, -108.49229431152344, -114.96744537353516, 34.49742126464844, 132.8310089111328, 72.27637481689453, 37.35628890991211, 31.604257583618164, -28.773700714111328, -68.51686096191406, -50.701622009277344, 91.96624755859375, 173.69956970214844, -4.721157550811768, 57.01106262207031, -66.28034973144531, -29.15747833251953, -3.313067674636841, 32.16904830932617, 79.0628433227539, 65.90471649169922, 71.20750427246094, -106.07457733154297, -125.48381805419922, -90.9732894897461, 51.89409637451172, 70.30797576904297, 77.62919616699219, 118.54649353027344, 138.2720489501953, 9.71623420715332, -21.00396728515625, 157.97642517089844, 213.3696746826172, 106.76276397705078, -114.74137878417969, -48.708953857421875, -95.18053436279297, -12.628454208374023, -163.48219299316406, 103.23490905761719, 47.683372497558594, -35.30691146850586, 128.66006469726562, 13.763412475585938, 27.162309646606445, 33.09520721435547, -34.62531280517578, -124.22268676757812, 41.626155853271484, -10.950889587402344, -193.42538452148438, -60.539791107177734, -42.38994216918945, 60.061561584472656, 13.437835693359375, 83.32249450683594, 49.943695068359375, -22.128698348999023, -147.75852966308594, 102.09638977050781, -80.10880279541016, 55.15941619873047, 78.75939178466797, -74.2948226928711, -95.20292663574219, 14.264643669128418, -28.48189926147461, 42.022029876708984, 97.1072998046875, -112.83817291259766, -71.38400268554688, 251.18597412109375, -30.73908042907715, 141.82765197753906, 21.206958770751953, -134.3480682373047, -57.152828216552734, 60.48881149291992, -87.96365356445312, 75.6226577758789, 48.59706497192383, -71.64905548095703, -38.018714904785156, -66.4503402709961, 19.756837844848633, 227.583740234375, 61.85090637207031, 45.97764587402344, 65.93265533447266, -91.01351928710938, -118.17280578613281, 169.74261474609375, -54.399295806884766, 2.691453456878662, 109.05756378173828, -84.32083892822266, 41.10807800292969, 20.614164352416992, -62.58403778076172, 21.052160263061523, -203.417724609375, -81.5355453491211, 189.14846801757812, -113.47997283935547, 3.674497127532959, 18.60194206237793, 140.19102478027344, -34.46710968017578, 53.21684265136719, 74.5940933227539, 4.730027675628662, -44.606956481933594, 252.1560516357422, 21.843629837036133, 36.772743225097656, 119.77729034423828, -167.26004028320312, 16.666126251220703, 143.2070770263672, 91.72114562988281, -172.98851013183594, 79.42549133300781, 127.18779754638672, -147.13963317871094, -19.567720413208008, -111.2354736328125, -7.324334144592285, 146.7061767578125, -5.238177299499512, -17.68849754333496, 12.544841766357422, -67.581298828125, 3.2841851711273193, -132.5377197265625, -127.9206771850586, 118.49537658691406, 156.9043426513672, 279.6027526855469, -50.35671615600586, 40.33116149902344, -40.450775146484375, 137.7367706298828, -56.75932693481445, -222.41110229492188, 8.324782371520996, -153.7400665283203, 74.96113586425781, 84.26847839355469, -43.11073684692383, 42.36182403564453, 55.1391716003418, -85.87797546386719, 14.618000984191895, 148.809326171875, 57.21510696411133, -186.6562957763672, -149.1580810546875, -61.636653900146484, -72.91288757324219, 195.62855529785156, -75.8382797241211, 2.6237502098083496, -43.62729263305664, -115.37664031982422, -114.66275024414062, -56.33932113647461, -147.26829528808594, -131.99623107910156, -164.15611267089844, -62.66104507446289, 32.93864822387695, 52.89802551269531, 0.13490237295627594, 8.466144561767578, 24.296863555908203, 79.55305480957031, 196.4160614013672, -143.58047485351562, -5.304797649383545, -67.53721618652344, -108.24530029296875, 143.02415466308594, 6.265860080718994, 54.15238571166992, 120.86297607421875, 69.13130950927734, -148.65289306640625, 56.9454345703125, 76.20674896240234, 145.1922149658203, 16.807310104370117, -14.135932922363281, 70.77713775634766, 156.2440185546875, 52.668121337890625, -56.14021682739258, -95.77469635009766, 147.23486328125, 21.449243545532227, -11.162566184997559, -44.97433090209961, -64.36251068115234, 159.1399383544922, 200.9212646484375, 46.43824005126953, -72.82096862792969, -33.009700775146484, -38.65144729614258, -72.89742279052734, -23.43790054321289, -206.2693328857422, 39.9869499206543, -181.02870178222656, -45.619869232177734, 114.61377716064453, 104.66590881347656, 71.70097351074219, -9.358063697814941, 126.23673248291016, 68.0370101928711, 25.706764221191406, -174.63343811035156, 70.94097900390625, -265.570556640625, 64.92984771728516, 150.9900360107422, -148.8905487060547, 83.51528930664062, 23.282644271850586, 11.003581047058105, 61.99315643310547, -143.93582153320312, -54.708045959472656, 22.371429443359375, 22.256444931030273, -162.33920288085938, 64.1784896850586, 117.22572326660156, 56.449337005615234, -59.185455322265625, -29.96283721923828, -38.55027389526367, -86.4108657836914, -148.51771545410156, 54.302127838134766, 26.277029037475586, -39.61509323120117, 116.53437042236328, -105.68094635009766, -57.21661376953125, -2.447547674179077, 210.46029663085938, -89.00365447998047, 80.7502212524414, 104.74837493896484, -10.289461135864258, -74.44872283935547, -41.95629119873047, 46.76630401611328, -268.1089172363281, 161.9183807373047, 10.40463924407959, 100.33982849121094, -80.69640350341797, 126.97142791748047, -98.10017395019531, -120.35772705078125, 0.4557126760482788, 190.66270446777344, 109.1678237915039, 33.79460144042969, -129.3922576904297, 88.83380126953125, -0.529170036315918, -175.18777465820312, -171.2587432861328, 16.3038387298584, -9.06240463256836, 8.642074584960938, -19.58210563659668, 4.536421775817871, -140.66734313964844, 0.6051309108734131, 39.110103607177734, 108.25068664550781, 46.95269775390625, 20.056533813476562, -10.79420280456543, 37.468994140625, -79.525146484375, -10.399130821228027, -37.1584587097168, -44.330848693847656, 60.93115997314453, -78.5283203125, 87.42951202392578, -55.648319244384766, -121.80574798583984, 23.210697174072266, 129.9512481689453, 136.65841674804688, -102.77336120605469, -51.277793884277344, -79.49263763427734, 12.154065132141113, 210.2626190185547, -130.7667236328125, -66.17797088623047, 176.7103729248047, -217.37118530273438, 225.5693359375, -106.73540496826172, -84.84032440185547, 97.08932495117188, 47.64655303955078, -58.33768844604492, -285.92718505859375, -28.07609748840332, -60.14509963989258, -93.22145080566406, -55.808589935302734, -188.1761932373047, -105.118896484375, -99.99222564697266, -99.51390075683594, -23.546030044555664, -9.736481666564941, -132.08045959472656, 3.751011371612549, -74.670654296875, -168.6985626220703, -7.823362827301025, 94.6031265258789, -67.04940795898438, 14.921098709106445, -94.8861083984375, -136.54298400878906, -30.278181076049805, -257.6714782714844, -9.832094192504883, -25.772422790527344, 13.126310348510742, 68.69042205810547, -8.542712211608887, 244.30035400390625, -138.152587890625, 101.0914306640625, -109.90721130371094, -52.0383415222168, 62.10251235961914, 101.50772094726562, -26.06612777709961, 74.29084777832031, -204.13601684570312, 43.86259078979492, -171.62391662597656, -62.47623825073242, -30.070314407348633, -52.82707977294922, 65.49547576904297, 77.03357696533203, 130.3467559814453, 95.93672943115234, 16.090129852294922, -184.6458740234375, 27.826274871826172, -37.41249084472656, -159.7940216064453, 229.7454071044922, 14.894550323486328, -146.25082397460938, -175.1838836669922, -109.71502685546875, -64.81035614013672, -11.783275604248047, -128.78619384765625, -11.678853988647461, -3.6054723262786865, 190.7154541015625, -23.1117000579834, -85.0740737915039, 33.04744338989258, 32.41695785522461, -25.696826934814453, 11.890968322753906, 16.798240661621094, 31.953943252563477, 293.41015625, -65.19293975830078, 12.678229331970215, -159.89431762695312, 24.51847267150879, 165.7533416748047, 211.2159423828125, 52.7344856262207, -39.03883743286133, 0.7541370391845703, 65.7318115234375, 121.82915496826172, 0.6733842492103577, -48.04310989379883, -21.016738891601562, -247.07298278808594, -156.41510009765625, 80.41439819335938, 137.73736572265625, 131.14173889160156, 80.64337158203125, -77.81050109863281, -87.13681030273438, -59.575862884521484, -58.99972915649414, -53.52121353149414, -8.334521293640137, -72.26793670654297, 30.08954429626465, -78.49071502685547, 50.49628448486328, -163.45458984375, 191.2908172607422, -7.873251914978027, 110.00009155273438, 38.05326461791992, 90.94441986083984, 33.452999114990234, -4.257833957672119, 68.73087310791016, 160.0230712890625, 16.647544860839844, 30.93777847290039, -124.26563262939453, 62.868160247802734, 132.65792846679688, 30.01569175720215, 138.77835083007812, 112.36923217773438, 23.989669799804688, -22.182876586914062, -39.182220458984375, -66.60348510742188, -13.027912139892578, 54.51162338256836, -29.879911422729492, 205.34010314941406, 6.732757091522217, 50.64035415649414, -13.987546920776367, -98.15133666992188, 51.25092697143555, 19.526159286499023, 45.128501892089844, 139.814697265625, -93.63887786865234, 55.83919143676758, 190.95867919921875, 129.9073486328125, 85.42481994628906, 124.80323028564453, -43.0555419921875, -93.186767578125, 126.40409851074219, 4.835736274719238, 32.9638786315918, -55.70623016357422, 47.85968017578125, -194.4393768310547, -130.8553009033203, -53.39511489868164, -208.50079345703125, -5.104440212249756, -92.63933563232422, -66.84150695800781, -2.631364345550537, 57.6502685546875, -60.237571716308594, -28.159170150756836, 122.00139617919922, -151.28628540039062, -78.82461547851562, 173.6009521484375, -76.86837005615234, -16.891250610351562, -114.27413940429688, 11.204567909240723, -36.852081298828125, 75.5960693359375, -127.95604705810547, 178.8544158935547, 1.921722412109375, 19.936756134033203, 157.36172485351562, 34.26166534423828, 269.7705078125, -93.05880737304688, -11.489335060119629, 37.87984085083008, 138.97024536132812, -75.41962432861328, 57.903907775878906, 98.57234191894531, 134.84764099121094, 41.28540802001953, 20.552465438842773, 56.78221893310547, -29.67620849609375, 218.46975708007812, -92.6995620727539, 47.38700485229492, 50.32997512817383, -69.1796875, -247.8299560546875, -12.864790916442871, 35.04706954956055, -58.1628303527832, 80.90667724609375, -23.766225814819336, -89.01152038574219, 22.451963424682617, 146.35516357421875, 54.99188995361328, -85.91734313964844, 78.22176361083984, 19.689340591430664, 67.1626205444336, -31.3348388671875, -22.235313415527344, -15.553786277770996, 33.41448211669922, -0.5074082016944885, 2.568159341812134, -79.62397003173828, 102.51636505126953, 62.824039459228516, 133.988525390625, -39.67363739013672, 147.86961364746094, -75.6349105834961, 63.24169158935547, -152.72918701171875, -22.075136184692383, -64.50886535644531, -84.60258483886719, -14.404191017150879, 25.806129455566406, -56.57461929321289, 19.57451629638672, 24.315698623657227, -71.5274887084961, -64.21187591552734, 81.37378692626953, 59.4971809387207, 0.8683660626411438, 21.33241081237793, -6.6817474365234375, -90.25718688964844, -191.4691925048828, 20.84417152404785, 142.08206176757812, 87.2985610961914, 167.36900329589844, 11.661763191223145, -14.987112998962402, -177.3857879638672, -98.3301010131836, 122.62316131591797, 52.8055534362793, 22.620311737060547, -64.93909454345703, -19.26439666748047, 20.86186408996582, -119.64790344238281, 62.325775146484375, 142.76434326171875, 61.25880813598633, -25.091157913208008, 197.2968292236328, 185.976806640625, -381.1659240722656, -94.90697479248047, -8.553364753723145, 183.9263458251953, 67.8285140991211, 84.15499114990234, 7.09318208694458, -161.93667602539062, 137.48464965820312, 74.0383071899414, -10.715356826782227, 95.4852523803711, 50.60359573364258, -170.78077697753906, 8.343501091003418, 15.565823554992676, -113.55899047851562, -49.48588943481445, -52.7419319152832, 19.03215789794922, 1.5036263465881348, 14.035506248474121, -8.692008018493652, -37.212249755859375, -126.64405059814453, 100.32288360595703, -116.70049285888672, 90.04623413085938, 33.08319091796875, -288.571044921875, 18.210994720458984, 7.105884075164795, 78.09556579589844, -50.32645034790039, 22.83692169189453, -23.08822250366211, 107.11609649658203, 136.69091796875, 97.8862533569336, 56.05657196044922, 29.202781677246094, -85.3956527709961, 117.18968200683594, 58.46782684326172, 10.866809844970703, 70.00033569335938, -59.711307525634766, 154.8634490966797, 61.47864532470703, 147.69219970703125, -156.5861053466797, -11.6063814163208, -37.00828552246094, 104.33599090576172, -46.00247573852539, 67.67868041992188, 185.9069061279297, -65.73333740234375, 15.594928741455078, 223.0061492919922, -25.334386825561523, 16.84375762939453, -99.58689880371094, -77.97017669677734, 214.3880615234375, 76.97817993164062, -80.10049438476562, -39.12504577636719, -66.78170776367188, -47.4874267578125, -135.3486328125, -122.80665588378906, 102.37779998779297, 108.06373596191406, -112.619873046875, 139.06199645996094, 133.29791259765625, -18.239166259765625, 19.77052879333496, -95.7210922241211, 41.337528228759766, -2.2219207286834717, 6.206672191619873, 145.8738250732422, 39.631324768066406, -43.377227783203125, 7.028112888336182, 118.9296646118164, -127.95349884033203, -37.27252960205078, -80.71539306640625, 31.16950225830078, 147.9925079345703, 11.461524963378906, 11.550251007080078, 15.061419486999512, 100.0810775756836, 16.085107803344727, -14.270926475524902, -26.883848190307617, -111.07007598876953, 139.90370178222656, -42.53879928588867, -31.303497314453125, 62.72138214111328, 79.3944320678711, 56.42881774902344, 36.53321075439453, -55.365234375, 158.05734252929688, 36.5119514465332, -18.778425216674805, 25.70547103881836, 42.4819221496582, 144.2259063720703, 166.1806640625, -31.343975067138672, -240.03318786621094, 5.965911388397217, -107.01453399658203, -6.8325605392456055, 148.42616271972656, 97.2879409790039, -171.9479217529297, -119.41337585449219, -40.97218322753906, -81.45588684082031, -95.19559478759766, -92.14923858642578, -212.67330932617188, 172.7352294921875, -53.326087951660156, 3.032439947128296, -28.399499893188477, 119.48985290527344, 205.5563507080078, 27.787006378173828, -1.998477578163147, 23.897981643676758, -235.24632263183594, -147.67086791992188, -179.0726776123047, 199.45481872558594, 68.53404235839844, 131.65589904785156, 112.61285400390625, 95.87539672851562, -161.7303009033203, 26.38582420349121, 58.60938262939453, 6.385701656341553, 102.21803283691406, 149.36256408691406, 4.681522369384766, -58.806427001953125, 117.52989959716797, -92.53668212890625, -72.74905395507812, 115.8265380859375, -16.164108276367188, -4.383919715881348, 68.82749938964844, 62.04740905761719, 384.9456481933594, -0.9047340154647827, 78.2529067993164, -195.7887725830078, 33.27698516845703, -24.014711380004883, 122.00694274902344, -187.8706512451172, 223.05026245117188, -127.0390853881836, 132.90760803222656, -53.360931396484375, -34.74494171142578, 80.47051239013672, -112.84141540527344, -23.110517501831055, -124.42777252197266, -128.7670135498047, 18.405189514160156, 102.75935363769531, -69.40233612060547, -81.71844482421875, -11.358094215393066, -253.5811309814453, -80.70148468017578, 137.89559936523438, 21.80359649658203, -111.98565673828125, 63.28763961791992, 96.96501159667969, -42.62018966674805, 33.623695373535156, 242.58226013183594, 41.89684295654297, -71.17501068115234, -125.0984878540039, 76.5042495727539, 200.45533752441406, 43.16362380981445, 188.66024780273438, -162.30711364746094, 55.292335510253906, 38.35389709472656, 93.48799896240234, 142.42774963378906, -76.61314392089844, -100.80332946777344, 23.5473690032959, -1.0815880298614502, -193.33485412597656, -109.30715942382812, 125.21727752685547, 42.29010009765625, -44.977813720703125, -7.734110355377197, 74.83704376220703, 85.58588409423828, -112.54583740234375, -14.02076244354248, 276.5707702636719, -3.8572683334350586, -4.815195560455322, -251.7522430419922, -24.176246643066406, -250.82647705078125, 7.754870891571045, -66.95735168457031, -18.705036163330078, -72.19650268554688, 46.83317565917969, 59.85889434814453, 169.77911376953125, -161.28834533691406, 11.399961471557617, 122.20335388183594, 78.84876251220703, 151.36077880859375, -261.00732421875, 57.6780891418457, -78.62567138671875, -23.414371490478516, -20.728057861328125, 191.78648376464844, 48.99427032470703, -85.40489959716797, 49.969539642333984, -28.105796813964844, 167.1409454345703, -33.198177337646484, 105.25506591796875, 60.40558624267578, 10.5452880859375, -66.48503112792969, -203.1096649169922, 57.81850814819336, -21.20012855529785, 93.4993667602539, 95.39846801757812, 9.087475776672363, 51.31747817993164, 40.47457504272461, -65.22702026367188, -138.7847442626953, 14.378074645996094, -22.015371322631836, -10.421591758728027, -194.85440063476562, 51.215904235839844, 1.5611995458602905, -30.979509353637695, 75.83926391601562, -76.91839599609375, 58.076026916503906, 253.06674194335938, 83.2664566040039, 180.98141479492188, -100.25008392333984, 17.990894317626953, 215.11741638183594, -107.21577453613281, -74.34235382080078, 206.3085479736328, 164.86402893066406, -186.30538940429688, 82.91099548339844, -233.2445831298828, 68.66012573242188, 147.47613525390625, 61.19411849975586, 118.66075897216797, -26.118061065673828, 198.08892822265625, 48.48375701904297, 20.461063385009766, -75.2867431640625, -32.378074645996094, -37.37554931640625, 50.64472961425781, -87.28401184082031, 89.37931823730469, 131.19895935058594, -241.6605682373047, 32.7830924987793, 41.10788345336914, -69.04850006103516, -18.124486923217773, 35.351417541503906, 111.0458984375, 4.214699745178223, 111.92642211914062, 49.11262130737305, -50.52476119995117, -76.7703857421875, -150.03005981445312, 58.58180236816406, 81.93961334228516, 32.749271392822266, -126.22037506103516, 127.01936340332031, 1.3988642692565918, 42.903568267822266, 2.8782169818878174, 64.58601379394531, 127.2832260131836, -91.75511169433594, -20.09044647216797, 57.798828125, -51.840267181396484, 32.496036529541016, 30.933170318603516, 6.0725531578063965, 27.546810150146484, -49.09061813354492, 128.53762817382812, 202.139892578125, 15.344025611877441, -12.713264465332031, -101.69371795654297, 101.11361694335938, -231.1368865966797, -69.58269500732422, 81.34345245361328, -62.177921295166016, 161.47781372070312, -2.810145854949951, -148.03404235839844, -68.968505859375, 116.13725280761719, 228.99317932128906, 71.03922271728516, -34.016780853271484, 61.282806396484375, -238.02964782714844, 92.55342864990234, -80.10022735595703, -90.19746398925781, -23.47532081604004, 31.71319007873535, 30.949960708618164, -87.02655029296875, 168.56800842285156, -45.70194625854492, -14.414386749267578, -33.99183654785156, -151.52117919921875, -76.44208526611328, -16.121679306030273, 169.8921661376953, -27.569263458251953, -345.6743469238281, 128.9565887451172, 181.1880645751953, 3.778679370880127, -125.89164733886719, 119.34796905517578, -107.61923217773438, 26.30170249938965, -8.554950714111328, 77.10897827148438, 106.7921142578125, 209.1556396484375, 92.76145935058594, -76.12578582763672, -13.859135627746582, -80.8019790649414, 72.92825317382812, 47.182098388671875, -7.308086395263672, -149.7367706298828, 150.03440856933594, 88.99436950683594, 69.30905151367188, 9.973669052124023, 154.74420166015625, -18.47409439086914, 107.99402618408203, 69.27136993408203, -24.588577270507812, 87.9326400756836, -81.63728332519531, 75.8221435546875, 6.741669654846191, 102.87136840820312, 96.57937622070312, 41.900856018066406, 136.86341857910156, -70.0930404663086, -30.459863662719727, 129.850341796875, -194.8455810546875, 36.06521987915039, -32.96741485595703, 33.34673309326172, -89.05437469482422, 130.294189453125, -40.54121780395508, 174.59584045410156, 21.083717346191406, 29.585893630981445, -207.5031280517578, 133.10166931152344, -35.62961196899414, -73.36257934570312, -202.753173828125, -169.77268981933594, 107.68091583251953, 98.59091186523438, -4.354550838470459, 125.53474426269531, -103.71250915527344, -43.038204193115234, -143.82376098632812, 12.915475845336914, 73.65620422363281, 61.94618606567383, -30.778722763061523, 74.44428253173828, 58.91667938232422, -76.59062194824219, -17.41777992248535, 5.521946430206299, -67.75518035888672, -187.7568359375, 18.326738357543945, 0.19489987194538116, -77.91468048095703, 31.9492130279541, 65.53874206542969, 12.606836318969727, 27.212623596191406, -27.475805282592773, -117.80404663085938, 85.5148696899414, -118.33512115478516, 73.72494506835938, -76.16084289550781, -129.05343627929688, 190.46621704101562, -17.5023136138916, -107.59233856201172, -190.47254943847656, -44.52396774291992, 97.27224731445312, 66.36234283447266, -29.887990951538086, -43.17366409301758, -60.7168083190918, 25.818382263183594, -47.37263870239258, -77.66861724853516, -142.53329467773438, 187.57557678222656, -168.68585205078125, 39.308349609375, 74.42671966552734, 53.441551208496094, 91.8228988647461, 73.56182861328125, -32.015281677246094, 78.94550323486328, 8.000813484191895, -21.548473358154297, -200.7693634033203, -162.4130401611328, -107.0595474243164, 59.150699615478516, 127.97088623046875, 97.12615203857422, 4.594511032104492, -37.744972229003906, -4.350296974182129, -8.213467597961426, -27.599332809448242, -100.89933013916016, -55.44371032714844, 18.65968894958496, 57.20610427856445, 103.34327697753906, -27.34310531616211, -69.13633728027344, -63.846744537353516, 51.64036560058594, -56.35775375366211, -78.63935089111328, 35.6837272644043, 132.14898681640625, 140.21609497070312, -4.510819911956787, 31.253070831298828, -92.2613754272461, -155.0415802001953, 180.86724853515625, 39.54853439331055, -89.43411254882812, -11.282844543457031, -102.10777282714844, -37.3991584777832, 17.38738441467285, -49.570045471191406, 35.10899353027344, 27.764541625976562, 153.14138793945312, -180.1162109375, -59.79447937011719, -135.37228393554688, -36.379364013671875, 53.753902435302734, 152.6332550048828, -13.23034381866455, 110.69564056396484, -15.616302490234375, -130.30218505859375, 70.95952606201172, 10.345922470092773, -138.40792846679688, 100.62083435058594, 192.57852172851562, 30.07815170288086, 186.2805633544922, -137.55685424804688, -41.40510177612305, -20.001379013061523, -34.240028381347656, 51.90984344482422, -10.26675796508789, 42.919830322265625, -31.239131927490234, -3.854870557785034, 26.486309051513672, -171.55667114257812, 104.85623168945312, -146.8729248046875, 28.024328231811523, 78.44158172607422, 7.742249011993408, -167.76268005371094, 53.55592727661133, 194.8976593017578, -29.12593650817871, 73.86458587646484, 45.98585510253906, -76.44303131103516, -135.8836669921875, 63.638572692871094, -3.2403159141540527, -101.68850708007812, 137.79055786132812, -71.57628631591797, 10.562018394470215, -87.81885528564453, 85.72721862792969, -34.258399963378906, -47.19162368774414, 4.421635627746582, -63.804176330566406, 120.56173706054688, 50.63096237182617, 135.35186767578125, 169.06097412109375, -180.48565673828125, -52.565582275390625, -152.4644775390625, 22.54450035095215, 112.65607452392578, 78.86823272705078, 90.02076721191406, 83.96652221679688, -110.36778259277344, 85.91316986083984, -140.1864013671875, -101.81908416748047, 194.1139678955078, -29.80954360961914, -95.4389877319336, -97.05638885498047, -49.41474914550781, -107.4837417602539, -180.82766723632812, -56.769710540771484, -38.63046646118164, -56.74357604980469, 114.11925506591797, -155.64486694335938, 108.87602996826172, -75.14283752441406, -79.0228500366211, -73.98774719238281, 28.146568298339844, 148.20654296875, 17.33683204650879, 2.709528923034668, 62.12239074707031, 45.250152587890625, -49.5381965637207, 30.640151977539062, -52.51451110839844, -19.27256965637207, -107.34599304199219, 26.101333618164062, -0.6977981328964233, -81.97966766357422, -15.221209526062012, -108.59221649169922, -67.22441864013672, 126.33415985107422, -59.404762268066406, 124.83264923095703, -67.10862731933594, -19.134796142578125, -175.2110137939453, 48.171295166015625, 176.17550659179688, -79.50394439697266, 76.12340545654297, 39.2022590637207, -77.25585174560547, 113.38617706298828, -77.5553207397461, -74.16783905029297, -17.867441177368164, 77.36780548095703, 180.5373992919922, -0.7643287181854248, -94.27714538574219, -73.709228515625, 129.76109313964844, 16.769485473632812, -234.5229034423828, 191.26416015625, -36.397621154785156, -141.06771850585938, -56.720176696777344, -86.61341857910156, -21.766164779663086, -62.030208587646484, -141.15293884277344, -17.350831985473633, 129.72474670410156, -128.65444946289062, 44.711936950683594, -10.883140563964844, 99.83662414550781, -69.15088653564453, -14.767723083496094, 34.21542739868164, -2.4394848346710205, 105.3564682006836, 48.605621337890625, 100.24665832519531, -121.6814193725586, 55.40397644042969, 42.061309814453125, 90.9146499633789, -71.75635528564453, -99.27796936035156, -41.3715705871582, 149.50546264648438, -38.74978256225586, -55.73775863647461, 33.27293014526367, -74.0331802368164, 3.5743494033813477, 196.36630249023438, -67.81560516357422, -143.07565307617188, -163.82147216796875, 99.85597229003906, 86.05502319335938, -104.59385681152344, -21.562150955200195, 83.22734832763672, -114.02635192871094, -92.5728988647461, -146.9958953857422, 1.3789851665496826, 107.38349151611328, 23.51504135131836, -33.44825744628906, 136.4429931640625, -27.16763687133789, -1.2805744409561157, -43.87211990356445, 31.497461318969727, -162.2853240966797, 216.47860717773438, -138.61788940429688, 92.28933715820312, 44.893218994140625, 193.6388397216797, -51.15767288208008, -89.4637680053711, -28.05840301513672, -43.135841369628906, 11.104656219482422, -57.326072692871094, 89.6121826171875, -150.70924377441406, 203.16282653808594, 13.421996116638184, 105.20323181152344, 25.011791229248047, 148.1864776611328, -92.74280548095703, -85.44062805175781, -25.98964500427246, -41.952205657958984, -86.01365661621094, 0.1385258585214615, 68.66524505615234, -68.62213134765625, 155.08152770996094, -153.56373596191406, 48.51593017578125, -88.72090911865234, 29.655168533325195, 15.110039710998535, -74.30460357666016, 26.643131256103516, -127.48395538330078, 153.7658233642578, -5.895756721496582, 33.60879898071289, -110.87932586669922, -122.33126068115234, -164.57546997070312, -42.979305267333984, 96.66886901855469, -3.174694061279297, -88.40404510498047, -56.310855865478516, 68.60194396972656, -93.7641830444336, 49.08395004272461, 21.730012893676758, 45.635196685791016, 118.21114349365234, -91.1920166015625, -221.48419189453125, 132.51722717285156, -31.926626205444336, -134.80081176757812, 134.8299102783203, 161.26112365722656, 196.42881774902344, 110.6439208984375, -87.16168212890625, -42.76243591308594, 59.77921676635742, 171.800537109375, -107.65147399902344, 14.559733390808105, -100.74150848388672, -79.92582702636719, 33.465702056884766, 72.5345458984375, 65.03633880615234, 114.58292388916016, 120.6187973022461, -5.013935089111328, -8.011396408081055, 134.55569458007812, -3.305361032485962, 97.40418243408203, 49.30768585205078, 77.52686309814453, -3.576136589050293, 203.28933715820312, 284.68670654296875, 11.201788902282715, -48.29790115356445, 0.12260649353265762, -18.65089225769043, 62.320960998535156, -107.08558654785156, 132.6559600830078, -186.71371459960938, -176.42259216308594, -21.897531509399414, -92.47896575927734, 45.77324295043945, -6.443700790405273, 41.97003173828125, 42.00262451171875, -1.1837633848190308, 47.688262939453125, -52.27366638183594, -88.51114654541016, 89.23041534423828, -52.43527603149414, 150.5911865234375, -49.9448356628418, -91.90884399414062, -48.33231735229492, -37.1602897644043, 23.704483032226562, -57.15781784057617, 137.0159912109375, -165.49951171875, -42.30255126953125, -105.65436553955078, -67.53241729736328, -171.13568115234375, 58.019161224365234, -23.719045639038086, 8.150365829467773, -3.6146867275238037, 61.24974822998047, 127.19778442382812, 266.9438781738281, -115.26238250732422, 19.681581497192383, 107.2342758178711, -135.29925537109375, -43.33266830444336, -56.09911346435547, 27.841251373291016, -140.3171844482422, -44.80656051635742, 17.746524810791016, -65.48914337158203, 26.948144912719727, 233.2088623046875, -28.139909744262695, 86.318359375, -7.188892841339111, 52.05933380126953, -47.3482666015625, 8.109415054321289, -137.22718811035156, -114.45345306396484, 135.29148864746094, 46.560367584228516, -37.5216064453125, -177.93402099609375, 74.51924133300781, 137.12625122070312, 127.21528625488281, -42.75644302368164, -115.53089141845703, 75.54200744628906, -123.66169738769531, 176.83070373535156, 83.50586700439453, -136.59149169921875, 105.15458679199219, -91.8475341796875, -5.55294942855835, -16.10074806213379, -74.18756103515625, -18.376522064208984, -24.283777236938477, 2.451026201248169, 97.55368041992188, -40.66746139526367, -158.718505859375, -1.316243052482605, -118.9704818725586, -139.70169067382812, 44.41105651855469, -1.2312523126602173, -43.87993240356445, 157.18136596679688, -43.83891296386719, -27.606916427612305, 35.81135940551758, 190.11874389648438, 66.26419067382812, -25.23267364501953, -44.71132278442383, -138.6322784423828, 31.10411834716797, 123.2376937866211, -213.32846069335938, 62.99040985107422, -78.20435333251953, -125.60567474365234, -102.84909057617188, -87.10450744628906, 86.9018783569336, -23.64500617980957, 67.55937194824219, -154.4014434814453, 6.561214923858643, -36.042964935302734, -127.03247833251953, -101.09833526611328, 7.135817527770996, -88.5333023071289, 6.68613338470459, 24.30963897705078, -20.860881805419922, 21.09490966796875, 46.49919128417969, 98.53065490722656, 11.255386352539062, 52.865325927734375, 90.96788787841797, 175.81381225585938, 5.555516719818115, -121.8865737915039, 31.876550674438477, 49.66734313964844, 21.66672134399414, 82.80252075195312, -89.36837005615234, -96.6463394165039, -60.58192443847656, -158.93203735351562, -105.61676025390625, -115.79740905761719, 92.50725555419922, -78.779541015625, -195.70529174804688, 45.77088928222656, 48.37741470336914, -48.16843795776367, -95.43350982666016, 39.02745056152344, -153.29205322265625, 74.84544372558594, 0.5627204775810242, 113.109130859375, 7.1817779541015625, 133.33145141601562, 37.693504333496094, -35.62545394897461, 114.2707748413086, -76.45042419433594, -69.26602935791016, -135.13278198242188, -57.10223388671875, 10.879496574401855, 52.87162399291992, -110.99847412109375, 45.67007064819336, -132.79429626464844, -96.14125061035156, 62.847145080566406, -41.59768295288086, -98.28015899658203, 0.9626505374908447, -46.884891510009766, -45.640071868896484, 132.73414611816406, -24.229944229125977, -60.391700744628906, -39.531978607177734, 107.84127044677734, -62.649810791015625, -49.45847702026367, -58.97804641723633, 114.2122573852539, 109.40109252929688, 83.92840576171875, 125.61766815185547, 21.556306838989258, 269.3401794433594, -76.52854919433594, -27.8103084564209, 60.1898193359375, -94.95545959472656, 30.60379409790039, 21.906869888305664, 3.916810989379883, 139.9300079345703, 44.885921478271484, 45.74816131591797, -129.63238525390625, 262.0499267578125, 163.700927734375, -169.4178009033203, 135.9703826904297, -98.34599304199219, -137.64158630371094, -59.42412185668945, -16.85335922241211, 23.869482040405273, 2.373619794845581, -77.07979583740234, 123.59686279296875, -150.1187286376953, 2.774254083633423, 109.55021667480469, 210.9154815673828, -8.038093566894531, -4.653173923492432, 7.842597007751465, -43.939422607421875, 72.66580200195312, -8.669018745422363, 220.02536010742188, -12.351988792419434, -111.46321868896484, -143.87879943847656, 150.70449829101562, 1.2861196994781494, -152.7058563232422, -37.647789001464844, -109.21109008789062, -50.86376190185547, -146.3688201904297, -42.38145446777344, -156.37266540527344, 174.23092651367188, 32.834693908691406, 5.702620983123779, 11.989287376403809, 52.4234504699707, -61.039913177490234, -128.97161865234375, -131.360595703125, -39.79240036010742, -51.82112503051758, 242.3037567138672, 220.97442626953125, 147.2550506591797, 22.049318313598633, 119.46576690673828, -0.13868902623653412, -0.4446882903575897, 99.25677490234375, -83.93170928955078, 48.46476364135742, 6.686740398406982, 26.68482780456543, 51.01777648925781, -43.52735137939453, -187.17613220214844, 57.90966796875, 47.453189849853516, -69.05228424072266, -13.13656997680664, -102.34046936035156, -17.01089096069336, 91.11788177490234, -4.852103233337402, 123.8338851928711, -49.81075668334961, -6.721348285675049, -288.8471984863281, 50.675506591796875, 40.39999008178711, -141.00071716308594, -5.707014560699463, -84.74337005615234, 8.941572189331055, 103.90925598144531, -95.60987854003906, 77.25398254394531, -107.29437255859375, 57.14625930786133, -57.02464294433594, 63.68205261230469, -88.38761901855469, -111.48334503173828, 36.257911682128906, 146.6085205078125, 117.33609771728516, 224.30224609375, 148.33609008789062, 114.8798599243164, 103.74446868896484, -109.50366973876953, 138.1734161376953, -70.61311340332031, 1.108148455619812, -119.09273529052734, -25.544416427612305, 116.86041259765625, -18.401975631713867, -109.54637908935547, -188.35693359375, 68.25790405273438, -13.184151649475098, 1.6411818265914917, -42.302154541015625, 38.75277328491211, -77.6363296508789, 45.26363754272461, 12.729682922363281, -11.071228981018066, 133.51309204101562, 25.94771957397461, -130.7754364013672, 153.88128662109375, 121.28623962402344, -3.4091672897338867, 129.44955444335938, -8.116633415222168, 14.252655982971191, -194.84585571289062, 123.47779083251953, -68.32445526123047, -10.089895248413086, 27.74005126953125, -0.1205744817852974, 19.769779205322266, 52.286434173583984, 79.73373413085938, -34.04255676269531, -241.04188537597656, 10.78337574005127, 87.60798645019531, -0.2471165955066681, -57.3686408996582, 169.5343780517578, -64.84339904785156, -46.676597595214844, -60.55165100097656, 83.30763244628906, -124.23888397216797, -50.4294548034668, 1.4320452213287354, 100.12013244628906, 236.31642150878906, 118.55048370361328, 9.356505393981934, 80.51729583740234, -6.587432861328125, 26.05563735961914, -33.032657623291016, -10.967000007629395, -55.85409927368164, 158.65029907226562, 19.757068634033203, 128.4110870361328, -60.300193786621094, 92.56312561035156, -50.15477752685547, 59.0751953125, -55.26022720336914, -20.078115463256836, -58.6055908203125, -42.52938461303711, 36.3918571472168, -248.0529327392578, 156.62408447265625, 73.2342758178711, -34.77294921875, 246.9615020751953, -30.482118606567383, 21.970129013061523, 211.2528076171875, 201.7976837158203, -63.49530792236328, 58.36006546020508, -22.166372299194336, 133.40525817871094, -90.00833892822266, 114.23434448242188, 20.266468048095703, -75.5971908569336, -0.5917834043502808, -78.53874206542969, -77.96306610107422, -54.777278900146484, 108.21846008300781, 36.019569396972656, 38.88739013671875, 26.90764045715332, 83.71524047851562, 81.61717987060547, 84.5557861328125, 75.62157440185547, 79.9348373413086, 10.91320514678955, 158.80181884765625, 79.98939514160156, -152.81649780273438, 191.29588317871094, 58.24617385864258, 83.046142578125, -26.071279525756836, -20.857275009155273, -17.956104278564453, -8.627958297729492, -10.683280944824219, -99.12657928466797, -15.83349323272705, -0.8935072422027588, -144.07977294921875, 42.35336685180664, 7.065670967102051, 36.040748596191406, -96.07217407226562, 93.561279296875, 133.40896606445312, 63.2636833190918, 172.14419555664062, -99.31938171386719, -9.174117088317871, 22.872940063476562, -10.865289688110352, 10.523626327514648, -204.33084106445312, -10.423095703125, 161.44932556152344, -68.87224578857422, 23.70140838623047, -26.041006088256836, 74.84580993652344, 176.39332580566406, 41.33713912963867, 41.76418685913086, -100.65550994873047, -35.33196258544922, -76.41633605957031, 92.47996520996094, 99.79618835449219, 56.3316535949707, -96.53278350830078, 50.34492492675781, 57.654632568359375, -164.92312622070312, 33.06023406982422, -10.996954917907715, -80.50077819824219, 186.63015747070312, -40.46406936645508, -16.0377254486084, -219.03292846679688, -78.35491943359375, 36.703067779541016, -65.766357421875, 76.34619903564453, 160.95916748046875, 98.88337707519531, -18.19038200378418, -201.1409912109375, 49.29345703125, -73.62993621826172, -131.73883056640625, 76.71893310546875, 74.95613098144531, 57.63560104370117, 118.03815460205078, 138.2580108642578, -13.19369125366211, -159.49557495117188, 143.49632263183594, -30.300012588500977, -68.191162109375, -150.7016143798828, -217.74728393554688, 11.65988826751709, 67.42792510986328, -57.179481506347656, -132.4607696533203, -103.6761245727539, 221.03416442871094, -111.07001495361328, 131.0505828857422, 8.17780590057373, -61.06867980957031, -135.13975524902344, -32.741817474365234, -52.5053596496582, -20.179336547851562, -14.041362762451172, 135.4940185546875, 191.38345336914062, 142.02088928222656, 124.82344055175781, -120.05210876464844, -10.286776542663574, -63.92171859741211, 24.799724578857422, 30.4515380859375, -109.51127624511719, -14.778621673583984, 13.681068420410156, 57.33964157104492, -44.9361572265625, -135.92959594726562, 14.802685737609863, 13.90670394897461, -43.02783203125, 35.71733093261719, -119.336181640625, -28.134897232055664, -236.7184295654297, 27.1658935546875, 102.89252471923828, -114.05751037597656, 198.0780792236328, 91.59341430664062, 24.698150634765625, -109.37039947509766, 119.05228424072266, -5.061017990112305, 55.97908401489258, 107.41976165771484, -5.145008563995361, -55.11945343017578, -22.712596893310547, -122.72148895263672, -53.03667068481445, 72.84357452392578, -75.88919830322266, -73.74091339111328, 5.899293899536133, 22.77033805847168, -38.06458282470703, 78.0740737915039, 66.21145629882812, 46.985538482666016, 4.672932147979736, -12.148534774780273, -5.847499370574951, -93.96256256103516, -30.544219970703125, 21.269092559814453, 2.9520628452301025, 205.8143768310547, 110.31135559082031, -2.2075371742248535, -44.676116943359375, -13.772544860839844, -105.74515533447266, -3.2243399620056152, 93.97276306152344, 2.810861587524414, 197.97474670410156, -8.71813678741455, 37.03694534301758, 68.20258331298828, 49.983524322509766, 87.38785552978516, 122.03194427490234, -105.15833282470703, -28.878448486328125, -47.81827163696289, 101.9622802734375, 9.224211692810059, 88.64065551757812, 63.226749420166016, -18.0282039642334, -8.687796592712402, -97.48047637939453, -0.3469812273979187, 105.93709564208984, 166.42982482910156, -55.97705078125, -119.17609405517578, -114.43153381347656, -42.714744567871094, 68.4087142944336, -120.4604263305664, -108.3517074584961, 8.900649070739746, -38.94493865966797, 111.5340576171875, -124.64510345458984, 34.34104537963867, -39.808509826660156, -71.38678741455078, -147.6669158935547, 247.1308135986328, 38.38093566894531, -7.045080661773682, 14.995182991027832, 23.521121978759766, 137.8245086669922, -2.833080291748047, 139.60206604003906, 68.89883422851562, -61.781185150146484, 83.45623779296875, -42.71372985839844, 134.7327117919922, -71.20340728759766, -87.09720611572266, 151.6953582763672, 4.954595565795898, -12.277640342712402, 28.802223205566406, 17.54427146911621, 143.80856323242188, 52.53822326660156, -44.03464889526367, -58.16487503051758, 3.9424126148223877, 217.68389892578125, 43.81687545776367, -77.48941802978516, -131.3447265625, -28.06819725036621, -34.8746223449707, 60.92431640625, 53.64946746826172, 42.04261016845703, -101.39408874511719, 30.957170486450195, -26.7485294342041, 43.539878845214844, -59.15374755859375, -15.534612655639648, 4.168017387390137, 116.71931457519531, -80.1541748046875, 1.120637059211731, 107.06959533691406, 88.33918762207031, 125.85343170166016, -178.25743103027344, -104.78326416015625, 95.6478042602539, -95.35076141357422, -112.04991149902344, -61.78498840332031, 121.0545654296875, -189.88172912597656, -6.827558517456055, 185.85960388183594, -94.56964874267578, 81.2914810180664, -96.32331085205078, -43.35383224487305, -190.50291442871094, -35.46397018432617, -136.6702423095703, -25.016870498657227, -51.85268783569336, 52.37380599975586, 50.59852600097656, 117.986572265625, -223.970458984375, 60.770137786865234, -131.98748779296875, -47.42615509033203, 33.32670211791992, -77.76390075683594, -32.314876556396484, 77.84199523925781, 111.22108459472656, -2.328381061553955, 246.3089141845703, 12.964834213256836, 39.24269104003906, -10.9973783493042, 173.27210998535156, 212.1106719970703, -19.142024993896484, -39.289031982421875, -241.80630493164062, -130.0183868408203, 126.39470672607422, 176.35488891601562, -10.458931922912598, -37.53542709350586, 224.6391143798828, 31.318281173706055, 27.536161422729492, 231.20970153808594, 39.85275650024414, 145.64923095703125, -138.7545166015625, 177.53623962402344, -110.3057861328125, -61.34088897705078, -49.935428619384766, -158.95880126953125, -131.01109313964844, 42.01863479614258, -37.538597106933594, 115.41825103759766, 20.8570613861084, -33.16978454589844, 165.0579833984375, 21.83205223083496, -7.299362659454346, -74.4756088256836, 56.58250045776367, 14.842021942138672, 126.9582748413086, -126.66908264160156, -369.3212890625, 31.781902313232422, 168.82423400878906, -9.154207229614258, 142.75047302246094, -75.4621353149414, -128.7045135498047, 129.20529174804688, 66.68890380859375, -0.31759896874427795, -5.730775356292725, -126.44799041748047, 12.456668853759766, 45.39244079589844, -90.9827651977539, 83.95626831054688, 68.30197143554688, 23.63018226623535, -205.62454223632812, -21.72179412841797, -27.62606430053711, -54.1097297668457, 15.315387725830078, 68.30772399902344, 23.403499603271484, 49.40119552612305, 74.44097137451172, -11.737713813781738, -99.70346069335938, -184.29991149902344, 79.0839614868164, -33.98674774169922, -143.02191162109375, -33.98830795288086, 206.64341735839844, 33.15664291381836, -20.95110321044922, 175.87745666503906, 11.337055206298828, 80.5439453125, -97.46285247802734, 29.374977111816406, 82.9701919555664, 41.241546630859375, 82.7891616821289, 34.53773498535156, -173.1571044921875, 7.179651737213135, -24.37760353088379, -108.07317352294922, -10.310431480407715, -40.57582092285156, 10.403605461120605, -64.41219329833984, 75.567626953125, 152.29466247558594, -7.627119541168213, -105.01659393310547, 102.91678619384766, -66.79985046386719, -139.88558959960938, -40.45047378540039, -25.92160987854004, 109.12239074707031, -10.59378719329834, -64.35304260253906, -16.06679916381836, -178.41177368164062, 124.0182876586914, -28.752323150634766, 63.00643539428711, 13.162182807922363, 72.37352752685547, -345.8623046875, -93.1850357055664, -107.13504791259766, 37.76361846923828, -56.021236419677734, 19.71659278869629, -64.03546142578125, 143.82017517089844, 33.01988220214844, 93.05024719238281, -11.57620906829834, -130.4505615234375, 2.644784450531006, 19.97650909423828, 55.3184700012207, -44.424251556396484, 150.52276611328125, 21.117095947265625, 52.195987701416016, -30.44927215576172, -272.3966369628906, -16.762540817260742, 193.38983154296875, -21.74471092224121, -72.69939422607422, -162.49440002441406, 64.89527130126953, -27.048446655273438, 105.96147918701172, -60.114463806152344, -69.23905944824219, -78.99905395507812, 119.1317367553711, -127.05355072021484, 1.8142720460891724, -116.42984008789062, -122.05967712402344, -110.73231506347656, 14.171895980834961, 27.61429214477539, 62.07518768310547, 127.38311767578125, 23.37085723876953, -89.20659637451172, 143.48251342773438, 135.85198974609375, -66.22467803955078, 19.75319480895996, -41.27466583251953, -37.633968353271484, 84.76428985595703, 27.270864486694336, 63.95728302001953, -23.59795570373535, 181.83551025390625, 5.801357269287109, -16.30048370361328, 90.91516876220703, 28.116147994995117, 158.565185546875, 160.48388671875, 244.1212921142578, 254.3595733642578, 6.565167427062988, -172.77903747558594, -8.047527313232422, 47.40647506713867, 40.379634857177734, -46.87976837158203, -96.85847473144531, 59.86932373046875, -3.45969295501709, -26.74108123779297, 60.32643508911133, 107.60433959960938, 43.635650634765625, -101.38029479980469, 186.6654510498047, 93.0758285522461, 111.39745330810547, -93.27989196777344, -139.1555938720703, 74.80496978759766, 46.82099914550781, 112.03042602539062, 34.483970642089844, 273.85504150390625, 168.1632080078125, -138.98033142089844, 25.63262367248535, -122.34994506835938, 75.13064575195312, -25.768152236938477, -56.70255661010742, 149.31724548339844, 87.70014953613281, -17.065187454223633, -150.70750427246094, 38.56369400024414, -52.51778793334961, 38.25940704345703, -80.6977767944336, -18.63538932800293, -57.18247604370117, -112.28981018066406, 10.344958305358887, 219.74136352539062, 152.435546875, -21.164899826049805, 47.200469970703125, -75.4276123046875, 9.732324600219727, 194.01876831054688, -76.35880279541016, 62.46511459350586, 87.64883422851562, -77.15653991699219, -56.82355880737305, -10.493293762207031, 89.60861206054688, -157.50411987304688, 38.98152160644531, 83.54351043701172, -72.06818389892578, 55.14303207397461, -37.67448043823242, -10.907622337341309, -60.7724723815918, -3.072610855102539, 26.762189865112305, 50.237335205078125, -25.851152420043945, -76.22863006591797, 50.87353515625, -105.59563446044922, -104.85774230957031, -116.07276916503906, -75.2728271484375, 35.58153533935547, 44.08699417114258, 118.00768280029297, -83.81583404541016, 31.293745040893555, -163.0803680419922, 98.88847351074219, 6.951200008392334, 58.168575286865234, -72.92375946044922, 116.8889389038086, 101.7128677368164, -25.231298446655273, -177.05966186523438, -105.18287658691406, 157.95391845703125, -196.93630981445312, -126.06536102294922, 6.897429466247559, 36.35887145996094, 10.59318733215332, -139.97824096679688, -55.42250061035156, -60.28959274291992, 5.95850944519043, 66.7007827758789, 228.737060546875, -92.89864349365234, -107.62805938720703, -20.86056137084961, -38.5720100402832, -159.76156616210938, 3.5881741046905518, -99.24113464355469, 85.25914001464844, -20.19251823425293, -31.416101455688477, 119.11345672607422, -192.26744079589844, 77.80667877197266, -109.27054595947266, 24.2324161529541, 12.400982856750488, -7.292073726654053, -0.1321956366300583, -29.912925720214844, 110.1136474609375, 16.521677017211914, 77.4257583618164, 67.87566375732422, 171.8643035888672, 100.77684020996094, 62.936283111572266, 63.22586441040039, -70.9911880493164, -179.04380798339844, -239.9732666015625, -37.95487976074219, -27.52369499206543, 123.31897735595703, -28.400949478149414, -63.536170959472656, -75.53155517578125, 128.3052978515625, 60.028377532958984, -218.13812255859375, 193.00286865234375, -86.18643188476562, -8.000112533569336, 165.9872589111328, -163.021240234375, 56.29082489013672, -1.7765791416168213, 48.969051361083984, -96.29902648925781, -0.34275317192077637, -61.47291564941406, -22.983905792236328, -63.12973403930664, 114.0304183959961, 109.7166748046875, 40.98754119873047, 57.08464813232422, -64.78478240966797, 101.24607849121094, 165.40972900390625, 147.36631774902344, 172.73165893554688, -17.746490478515625, 49.7524299621582, 5.633620738983154, 46.91865921020508, 35.96297836303711, 102.22933197021484, -22.44987678527832, -151.28182983398438, -8.255326271057129, 248.7720947265625, 52.168174743652344, -29.38998794555664, 66.26209259033203, 124.2317123413086, -124.97959899902344, -42.046627044677734, -249.40830993652344, 6.3281636238098145, 101.63360595703125, 34.986637115478516, -9.243144989013672, -153.43276977539062, -134.48666381835938, 61.758758544921875, -4.7386298179626465, 7.011724472045898, 12.974075317382812, 21.708362579345703, 48.62942123413086, 110.95185089111328, -51.29671859741211, 181.6363067626953, 83.73480987548828, 38.93544387817383, -100.00689697265625, -36.91574478149414, -121.59983825683594, -147.42552185058594, -73.96744537353516, 6.677351951599121, 101.681640625, -116.85607147216797, -37.251338958740234, -8.73884105682373, 45.348533630371094, 22.78186798095703, -21.020185470581055, -94.6783447265625, 21.78173065185547, -73.58320617675781, 7.489795207977295, -5.977948188781738, -142.1907958984375, 49.305503845214844, -90.81273651123047, 126.5750732421875, -54.33086013793945, 65.01234436035156, 119.52943420410156, -18.065391540527344, 40.628761291503906, 46.326480865478516, 29.83457374572754, 18.67612075805664, -59.527767181396484, -88.33515167236328, 113.76306915283203, 19.1153564453125, -56.145713806152344, -8.298151969909668, 15.982477188110352, -15.268486976623535, -74.09915924072266, 19.732093811035156, -20.236328125, -20.440004348754883, 174.9654541015625, -50.02069854736328, 55.78641128540039, 39.4251708984375, 35.85894012451172, -232.14219665527344, 184.79119873046875, -116.2949447631836, 20.373117446899414, -177.9963836669922, -118.88478088378906, -31.621923446655273, 148.8946533203125, 212.4730682373047, -17.679401397705078, -134.00352478027344, 20.364356994628906, -189.6896514892578, -98.44412231445312, -140.2454071044922, -51.79427719116211, -129.9166717529297, 76.07173156738281, -55.53530502319336, -22.400957107543945, 179.76837158203125, -144.72935485839844, -72.05078887939453, 27.48027992248535, -6.64536714553833, 172.582763671875, -142.70852661132812, -36.518245697021484, 13.533858299255371, -310.4639892578125, 138.49688720703125, 10.29988956451416, -3.194157838821411, 25.08713722229004, -146.40443420410156, 55.205894470214844, -55.165225982666016, -65.53683471679688, -98.02140045166016, -160.1590576171875, 78.43841552734375, 59.931312561035156, -20.672813415527344, 57.54839324951172, -15.109145164489746, 24.517589569091797, -51.987709045410156, -93.84567260742188, -44.1262092590332, -291.7567443847656, 135.34457397460938, -4.6090593338012695, 97.30261993408203, -140.6698760986328, 136.09996032714844, 115.25115966796875, 6.1326165199279785, -25.99357032775879, 69.23761749267578, -1.7405990362167358, 63.844940185546875, 92.90998840332031, -123.68634796142578, 48.200592041015625, 30.354537963867188, 90.6683578491211, 107.70630645751953, 71.24836730957031, -198.46051025390625, 155.73397827148438, -53.65243911743164, -154.18186950683594, 55.94825744628906, -24.041248321533203, 56.22928237915039, -47.81978225708008, -16.744081497192383, 20.64531898498535, 123.76236724853516, 11.68268871307373, 61.327667236328125, -7.74190616607666, 48.498512268066406, 115.74114990234375, -171.13607788085938, -108.60611724853516, -31.750831604003906, -234.13414001464844, -159.98980712890625, -166.2895965576172, -161.16067504882812, -88.84825134277344, 155.8357391357422, -77.97078704833984, 53.925350189208984, 120.18424987792969, 49.7126579284668, -62.13067626953125, 32.54302978515625, 88.41657257080078, -34.585411071777344, -148.06494140625, 1.9817407131195068, 141.16845703125, -229.0457000732422, 128.34117126464844, 3.7062511444091797, 54.73029327392578, -98.32511901855469, 25.808807373046875, -116.61841583251953, 44.76688003540039, -113.06216430664062, 145.21815490722656, 98.30554962158203, 195.11419677734375, -77.3908462524414, -108.34204864501953, 102.79473114013672, -122.45004272460938, -64.20740509033203, -54.70307922363281, -72.06439208984375, -64.33575439453125, -45.9802131652832, 196.89764404296875, 4.397463321685791, 104.32258605957031, -25.90999984741211, 22.98401641845703, 15.531810760498047, 53.19746017456055, -265.4798583984375, 54.9470100402832, 128.093505859375, 25.902925491333008, 0.9324001669883728, -110.90687561035156, 17.47260284423828, 66.27540588378906, 79.12644958496094, -41.942752838134766, 108.64540100097656, 183.07447814941406, 15.658559799194336, -0.7821279168128967, -51.59002685546875, 201.99859619140625, -83.09117889404297, -69.05723571777344, -45.4532585144043, 119.43447875976562, -5.352950096130371, -36.6868896484375, -49.825138092041016, 90.78758239746094, 2.1973519325256348, -53.47628402709961, 134.0431671142578, -11.542757034301758, 94.53255462646484, -11.242687225341797, -56.133087158203125, -120.13768005371094, 16.98134994506836, -83.35723876953125, -29.472309112548828, 95.16036224365234, -46.68800735473633, -47.14404296875, -27.172748565673828, -113.05032348632812, -5.289772987365723, 173.8265838623047, 63.4284553527832, -214.91455078125, -90.15086364746094, 135.14842224121094, -123.76026916503906, -158.98980712890625, -152.2015380859375, -8.914886474609375, -149.62942504882812, -28.093568801879883, -34.869117736816406, 46.56850814819336, 11.562976837158203, -112.56887817382812, 16.57921028137207, 77.34971618652344, -108.95672607421875, -154.80947875976562, -113.96328735351562, -17.85085105895996, 60.07872009277344, -40.500205993652344, -35.13987731933594, 63.060489654541016, 143.1681671142578, -128.5900115966797, 82.81066131591797, 24.788480758666992, 99.58756256103516, -102.70730590820312, -20.210918426513672, -93.65895080566406, -89.07572174072266, 4.387307167053223, -93.57489013671875, -46.814613342285156, 139.1734161376953, -77.255859375, -34.278385162353516, 12.804582595825195, -196.55679321289062, -45.81499099731445, 57.749839782714844, -95.31985473632812, 12.25662899017334, 70.728759765625, -86.88473510742188, -24.539833068847656, -173.2528533935547, -7.8277740478515625, -215.70574951171875, 237.6658172607422, -64.43354034423828, 272.64190673828125, -80.53898620605469, -96.30233764648438, 125.98233032226562, -43.78367233276367, -18.048198699951172, 92.4367446899414, 45.43251419067383, -115.31257629394531, 31.904375076293945, 171.18023681640625, -88.58056640625, 11.193565368652344, -94.80963134765625, -59.291709899902344, 27.659908294677734, 248.29742431640625, -23.449188232421875, -59.12302780151367, 35.91401290893555, 25.720970153808594, 178.5767059326172, -44.25886154174805, -84.31145477294922, 43.571746826171875, -197.43637084960938, 34.12089920043945, 9.308905601501465, 15.413491249084473, -37.06623458862305, -153.98284912109375, -118.09357452392578], "dtype": "float32"}], "e": 384.9456481933594}]'
import json, types
harness = types.ModuleType('harness'); exec(_HARNESS, harness.__dict__)
res = harness.grade(array_max, json.loads(_TESTS), 'cuda', *[1e-06, 1e-08])
for i, r in enumerate(res): print(f"test {i+1}: {'pass' if r['ok'] else 'FAIL  ' + r['why']}")
print(f"{sum(r['ok'] for r in res)}/{len(res)} passed")